# Korpus Batak Toba dan Indonesia: Alur Lengkap Pengolahan

Notebook ini menampilkan **seluruh pengolahan korpus dalam satu tempat**, dari hasil kerja dua narasumber penerjemah sampai korpus paralel final yang sudah dianotasi dan diadjudikasi oleh dua penilai.

Penelitian: *Digitalisasi dan Praproses Korpus Bahasa Batak Toba sebagai Fondasi Pengembangan Model Bahasa Lokal Berbasis Deep Learning* (Skema PDP, Universitas Pelita Harapan).

Notebook ini menggantikan `pipeline_korpus_multigenre_v2.ipynb` sebagai notebook utama. Notebook lama tetap disimpan sebagai arsip eksperimen penyelarasan ulang.

## Daftar isi

| Bagian | Isi | Tahap |
|---|---|---|
| 0 | Peta alur dan riwayat keputusan | |
| A | Persiapan, daftar input, dan konfigurasi | 0 |
| B | Korpus multi-genre: validasi, koreksi label, pembersihan, segmentasi, statistik | 1 sampai 5 |
| C | Korpus paralel tingkat kalimat: korpus v1, IAA, adjudikasi, penerapan | 6 sampai 9 |
| D | Statistik akhir dan paket rilis | 10 |
| E | Ringkasan dan keterbatasan | |

## Prinsip

1. **Setiap angka bisa ditelusuri ke file input.** Tidak ada perbaikan manual di luar notebook. Setiap koreksi tercatat di log.
2. **Parameter ditetapkan sebelum dijalankan.** Semua parameter ada di sel Konfigurasi.
3. **Titik simpan (checkpoint).** Setiap tahap menyimpan keluarannya ke Google Drive. Saat dijalankan ulang, tahap yang masukan, parameter, dan kodenya tidak berubah diambil dari checkpoint.
4. **Tidak butuh GPU.** Semua tahap berjalan di runtime CPU biasa dalam beberapa menit.

---
# Bagian 0. Peta alur dan riwayat keputusan

## Peta alur

```
 Dua narasumber penerjemah
 (Annotator #1: 64 dokumen, Annotator #2: 40 dokumen)
            |
            v
 Combined Annotator.xlsx (104 dokumen paralel)
            |
   +--------+-------------------------------------------+
   |                                                    |
   v  BAGIAN B (tahap 1-5, dijalankan di notebook ini)  v  DIBUAT SEBELUM NOTEBOOK INI (dibekukan)
 koreksi label, pembersihan,                         segmentasi + LaBSE + audit semi-manual
 segmentasi, statistik                                          |
   |                                                            v
   v                                               btb_parallel_sentence_v1.csv (1.505 pasangan)
 btb_multigenre.csv (rilis)                         + lembar Perlu_periksa (145 pasangan ditandai)
   |                                                            |
   |                                         BAGIAN C (tahap 6-9)
   |                                                            v
   |                                           Anotator A dan B menilai 145 pasangan
   |                                           -> rekonsiliasi label A -> IAA (kappa)
   |                                           -> adjudikasi 78 baris keputusan + 49 baris teks
   |                                           -> keputusan emas diterapkan ke korpus
   |                                                            |
   |                                                            v
   |                                           korpus paralel final (1.542 pasangan)
   |                                           doc_id dan genre disamakan dengan Bagian B
   |                                                            |
   +---------------------------+--------------------------------+
                               v
                BAGIAN D (tahap 10): statistik dan paket rilis
```

## Riwayat keputusan

Bagian ini mencatat perubahan strategi selama penelitian, supaya alasan di balik setiap tahap tidak hilang.

| No | Peristiwa | Keputusan dan alasannya |
|---|---|---|
| 1 | Dua narasumber menyalin dan menerjemahkan 104 dokumen ke Google Sheet | Gabungan keduanya, `Combined Annotator.xlsx`, menjadi satu-satunya input korpus multi-genre |
| 2 | Notebook lama (`01_build`, `02_eda`) menghasilkan `btb_multigenre.csv` dan angka 112.340 token | Belakangan diketahui angka itu dihitung dengan NLTK yang **menghitung tanda baca sebagai token**, padahal README menyebut tokenisasi berbasis spasi. Angka 520 dokumen Alkitab juga keliru (seharusnya 66 pasal, 1.594 ayat). Sekarang jumlah token dilaporkan dengan tiga metode (tahap 5) |
| 3 | Notebook segmentasi lama menyelaraskan kalimat dengan LaBSE, lalu dilakukan audit penyelarasan semi-manual | Menghasilkan korpus paralel v1 (1.505 pasangan) dan 145 pasangan berprioritas tinggi atau sedang di lembar `Perlu_periksa`. Sebagian langkahnya dikerjakan di luar notebook, sehingga di sini korpus v1 dipakai sebagai **input beku** (tahap 6) |
| 4 | Pemeriksaan ulang `Combined Annotator.xlsx` menemukan 12 label genre atau sub-genre salah, metadata contoh dari templat, dan URL di kolom yang salah | Diperbaiki sebagai langkah tercatat (tahap 2). Korpus paralel v1 masih memakai label lama, jadi labelnya disamakan di tahap 9 |
| 5 | Pipeline v2 mencoba menyelaraskan ulang seluruh korpus dengan LaBSE dan menyiapkan lembar review 675 baris | **Tidak dipakai.** Anotasi manusia akhirnya dilakukan pada korpus v1, sehingga hasil penyelarasan ulang tidak punya validasi manusia. Bagian itu dikeluarkan dari notebook utama |
| 6 | Dua penilai baru (A dan B), yang berbeda dari kedua penerjemah, menilai 145 pasangan di lembar `Perlu_periksa` | Penilai tidak menilai terjemahannya sendiri, sehingga kesepakatannya mencerminkan penilaian independen |
| 7 | Anotator A memberi `accept_unit` pada 123 baris, tetapi menyunting teks pada 92 di antaranya | A diminta merekonsiliasi label dengan aturan: kalau teks diubah, keputusannya bukan `accept_unit`. Aturan ini mekanis dan diperiksa ulang di tahap 7, beserta analisis sensitivitas bila aturan yang sama diterapkan ke B |
| 8 | Adjudikasi oleh pihak ketiga | 78 baris yang keputusannya berbeda diputuskan di lembar adjudikasi keputusan; 49 baris dengan keputusan sama tetapi teks koreksi berbeda diputuskan di lembar adjudikasi teks (tahap 8) |
| 9 | Keputusan emas diterapkan ke korpus v1 | Menghasilkan 1.542 pasangan. Notebook ini memeriksa bahwa hasilnya sama persis dengan `korpus_utama_FINAL_v3` (tahap 9) |
| 10 | Sub-korpus Alkitab | Dikeluarkan dari notebook ini. Sisi Indonesianya memakai Terjemahan Baru (1974) yang izin redistribusinya perlu dipastikan. Akan diolah di notebook terpisah |

---
# Bagian A. Persiapan

## Tahap 0. Persiapan lingkungan

**Tujuan:** menyiapkan lingkungan kerja yang sama setiap kali notebook dijalankan.

**Yang dikerjakan:**
1. Memasang pustaka (`openpyxl` untuk berkas Excel, `nltk` untuk pembanding hitungan token versi lama).
2. Menghubungkan Google Drive bila notebook berjalan di Colab.
3. Membaca konfigurasi, menyiapkan folder kerja, dan mendefinisikan fungsi bantu checkpoint.

## Daftar input

Siapkan folder kerja di Google Drive dengan susunan berikut. Hanya file-file ini yang dibutuhkan.

```
btb-alur-lengkap/
└── input/
    ├── Combined Annotator.xlsx
    ├── btb_parallel_sentence_v1.csv
    ├── anotasi/
    │   ├── anotator_A_asli.xlsx
    │   ├── anotator_A_rekonsiliasi.xlsx
    │   └── anotator_B.xlsx
    ├── adjudikasi/
    │   ├── adjudikasi_keputusan_78.xlsx
    │   └── adjudikasi_teks_49.xlsx
    └── pembanding/
        └── korpus_utama_FINAL_v3.csv      (opsional)
```

| File | Asal | Dipakai di |
|---|---|---|
| `Combined Annotator.xlsx` | Gabungan Google Sheet kedua narasumber | Tahap 1 |
| `btb_parallel_sentence_v1.csv` | Repositori GitHub, `data/experimental/`. Sama dengan `korpus_utama_revisi_penyelarasan_v1.csv` | Tahap 6 |
| `anotator_A_asli.xlsx` | `audit_penyelarasan_toba_v1-sapan.xlsx` | Tahap 7 |
| `anotator_A_rekonsiliasi.xlsx` | `audit_penyelarasan_toba_v1-sapan-REVISI` (unduh sebagai .xlsx) | Tahap 7 dan 8 |
| `anotator_B.xlsx` | `audit_penyelarasan_toba_v1-tebi.xlsx` | Tahap 7 dan 8 |
| `adjudikasi_keputusan_78.xlsx` | `adjudikasi_78_baris` (unduh sebagai .xlsx) | Tahap 8 |
| `adjudikasi_teks_49.xlsx` | `adjudikasi_teks_49` (unduh sebagai .xlsx) | Tahap 8 |
| `korpus_utama_FINAL_v3.csv` | `korpus_utama_FINAL_v3` (unduh sebagai .csv). Versi .xlsx juga diterima | Tahap 9, hanya untuk verifikasi |

Nama file boleh berbeda, asal disesuaikan di sel Konfigurasi.

**Catatan:** gunakan folder kerja baru, jangan folder `PUSTAHA - Pipeline Korpus Multi-genre` yang lama. Folder lama berisi keluaran penyelarasan ulang v2 yang tidak dipakai lagi dan bisa membingungkan.

In [ ]:
%pip install -q openpyxl nltk

In [ ]:
import os
import sys
import json
import math
import random
import shutil
import hashlib
import platform
import unicodedata
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

if DI_COLAB:
    drive.mount("/content/drive")

pd.set_option("display.max_colwidth", 80)
print("Berjalan di Colab:", DI_COLAB)
print("Python", platform.python_version(), "| pandas", pd.__version__)

## Konfigurasi

Seluruh parameter ada di satu sel ini. Setiap tahap mencatat sidik (hash) dari parameter, berkas masukan, dan kode yang dipakainya. Bila salah satunya berubah, tahap itu dan semua tahap sesudahnya otomatis dihitung ulang saat **Run all**.

| Bagian | Isi | Dipakai di tahap |
|---|---|---|
| `folder_kerja` | Lokasi folder kerja di Google Drive | Semua |
| `berkas` | Nama setiap file input | Semua |
| `paksa_ulang` | Kode tahap yang dipaksa dihitung ulang, misalnya `["s07"]` | Semua |
| `kolom_anotator`, `koreksi_metadata`, `isian_templat`, `normalisasi_label` | Pemetaan kolom dan koreksi label korpus multi-genre | 1, 2 |
| `segmentasi` | Aturan pemecahan kalimat | 4 |
| `iaa` | Kategori keputusan, aturan rekonsiliasi, dan angka kappa utama yang dilaporkan | 7 |
| `koreksi_pasca_adjudikasi` | Perubahan keputusan emas setelah lembar adjudikasi selesai, lengkap dengan alasan dan sumbernya | 8 |
| `peta_doc_id_v1` | Aturan pemetaan `doc_id` korpus v1 ke `doc_id` korpus multi-genre | 9 |

In [ ]:
KONFIGURASI = {
    "versi_pipeline": "3.0.0",
    "folder_kerja": os.environ.get("BTB_FOLDER_KERJA", "/content/drive/MyDrive/Datasets/btb-alur-lengkap"),
    "seed": 42,
    "paksa_ulang": [],
    "target_token": 100_000,

    "berkas": {
        "anotator_narasumber": "Combined Annotator.xlsx",
        "korpus_paralel_v1": "btb_parallel_sentence_v1.csv",
        "anotator_A_asli": "anotasi/anotator_A_asli.xlsx",
        "anotator_A_rekonsiliasi": "anotasi/anotator_A_rekonsiliasi.xlsx",
        "anotator_B": "anotasi/anotator_B.xlsx",
        "adjudikasi_keputusan": "adjudikasi/adjudikasi_keputusan_78.xlsx",
        "adjudikasi_teks": "adjudikasi/adjudikasi_teks_49.xlsx",
        "pembanding_final_v3": "pembanding/korpus_utama_FINAL_v3.csv",
    },
    "sheet_anotator": 0,
    "sheet_review": "Perlu_periksa",
    "kolom_anotator": {
        "Column 1": "nomor_anotator",
        "Date Collected": "date_collected",
        "Title": "title",
        "Text in Toba": "text_bt",
        "Text in Bahasa Indonesia": "text_id",
        "Genre Category": "genre",
        "Sub-Genre Category": "subgenre",
        "Original Author (Publisher or Person)": "original_author",
        "Source Link": "source_link",
        "Additional Notes (optional)": "notes",
    },
    "kolom_wajib": ["Title", "Text in Toba", "Text in Bahasa Indonesia", "Genre Category", "Sub-Genre Category"],

    "koreksi_metadata": [
        {
            "judul_mengandung": ["Jangan Membuat Masalah Kecil", "Ikigai"],
            "kolom": "genre",
            "nilai_baru": "Educational",
            "alasan": "Ulasan buku, tercantum di daftar sumber Educational (Book summary) pada panduan anotator. Label lama berasal dari isian contoh templat.",
        },
        {
            "judul_mengandung": ["Kota Cerdas Berkelanjutan", "Jangan Membuat Masalah Kecil", "Ikigai"],
            "kolom": "subgenre",
            "nilai_baru": "Book Summary",
            "alasan": "Ulasan buku, tercantum di daftar sumber Educational (Book summary) pada panduan anotator. Label lama berasal dari isian contoh templat.",
        },
        {
            "judul_mengandung": [
                "Analisis Literasi Artifical Intelligence",
                "Implementasi Teknologi Artificial Intelligence",
                "Metafora cinta dalam bahasa Batak Toba",
                "ANALISIS PENERJEMAHAN SEJARAH DAN BUDAYA",
                "Peran Usaha Mikro, Kecil Dan Menengah",
                "Dampak Ekonomi Digital Terhadap Pertumbuhan",
                "SIKAP ETIS UMAT KRISTEN",
                "Pendidik sebagai Pembimbing Rohani",
                "Kajian Doa Menurut Alkitab",
            ],
            "kolom": "subgenre",
            "nilai_baru": "Abstract",
            "alasan": "Abstrak artikel jurnal, tercantum di daftar sumber Educational (Abstract) pada panduan anotator. Label lama News Article tidak sesuai.",
        },
        {
            "judul_mengandung": ["Raja Martua Doli"],
            "kolom": "source_link",
            "nilai_baru": "https://www.hutapea.id/2019/06/torsa-torsa-raja-martua-doli-dohot.html",
            "alasan": "Kolom sumber berisi penanda [URL]. Tautan diambil dari daftar sumber Annotator #1 pada notebook panduan anotator.",
        },
    ],
    "isian_templat": {
        "original_author": ["Unknown", "Community", "Dr. Sari Dewi", "Local Council", "Tempo Magazine"],
        "notes": [
            "Needs cross-referencing with other Batak tales.",
            "Translation in progress for key sections.",
            "Compare with older versions.",
            "Used for online course.",
            "Awaiting full transcription.",
            "Part of media analysis project.",
        ],
        "date_collected": [
            "2025-10-01 00:00:00", "2024-11-20 00:00:00", "2025-01-05 00:00:00",
            "2025-03-22 00:00:00", "2025-07-10 00:00:00", "2024-12-01 00:00:00",
        ],
    },
    "normalisasi_label": {
        "genre": {"Historical/ Traditional": "Historical/Traditional"},
        "subgenre": {},
    },
    "placeholder_sumber": ["[URL]", "URL", "-", "n/a"],

    "segmentasi": {
        "pisah_setiap_baris_baru": True,
        "singkatan": [
            "dr", "drs", "dra", "ir", "prof", "hj", "h", "tn", "ny", "sdr",
            "yth", "dll", "dst", "dsb", "tsb", "tgl", "no", "hlm",
            "jl", "jln", "kel", "kec", "kab", "prov",
            "st", "sd", "smp", "sma", "smk", "pt", "cv", "tbk",
            "op", "ompu", "amang", "inang", "ama", "ina",
            "vs", "etc", "vol",
        ],
    },

    "subgenre_liturgi": ["Ritual Text"],

    "iaa": {
        "kategori": ["accept_unit", "revise_alignment", "revise_translation", "need_source", "fragment_policy", "exclude"],
        "penanda_pecah": "|||",
        "aturan_rekonsiliasi": "Bila teks koreksi berbeda dari teks asli tetapi keputusannya accept_unit, keputusan diganti revise_alignment bila koreksi memuat penanda pecah, selain itu revise_translation.",
        "kappa_utama": "rekonsiliasi_A",
        "bootstrap": 1000,
    },

    "koreksi_pasca_adjudikasi": [
        {
            "pair_id": "doc_0000_s013",
            "keputusan": "revise_translation",
            "teks_bt": None,
            "teks_id": 'Maka berkatalah menantunya itu, "Mencuri itu memalukan, Ibu, lagipula sudah menjadi hukum/kebiasaan bahwa makanan (hasil buruan) itulah yang harus diambil!',
            "alasan": "Lembar adjudikasi memutuskan need_source. Peneliti kemudian menerjemahkan ulang sisi Indonesia sehingga keputusan menjadi revise_translation, sebagaimana tercantum di korpus_utama_FINAL_v3.",
            "sumber": "Terjemahan peneliti (Feliks Victor Parningotan Samosir), dibuat setelah adjudikasi",
        },
    ],

    "peta_doc_id_v1": {
        "keterangan": "doc_id v1 mengikuti urutan berkas v1.1 (99 dokumen, enam bagian Agenda HKBP digabung menjadi dokumen terakhir).",
        "batas_agenda": 53,
        "geser_setelah_agenda": 6,
        "doc_agenda_v1": "doc_0098",
    },
}

random.seed(KONFIGURASI["seed"])
np.random.seed(KONFIGURASI["seed"])

ROOT = Path(KONFIGURASI["folder_kerja"])
DIR_INPUT = ROOT / "input"
DIR_OUT = ROOT / "keluaran"
DIR_META = DIR_OUT / "_meta"
DIR_RILIS = ROOT / "rilis"
for d in (DIR_INPUT, DIR_OUT, DIR_META, DIR_RILIS):
    d.mkdir(parents=True, exist_ok=True)

BERKAS = {k: DIR_INPUT / v for k, v in KONFIGURASI["berkas"].items()}
BERKAS_ANOTATOR = BERKAS["anotator_narasumber"]
print("Folder kerja:", ROOT)
for k, p in BERKAS.items():
    print(f"  {k:26s} {'ada' if p.exists() else 'TIDAK ADA':10s} {p.relative_to(ROOT)}")

## Fungsi bantu checkpoint

Setiap tahap dibungkus objek `Tahap`. Objek ini menghitung **sidik** dari empat hal: parameter tahap, isi berkas masukan (SHA-256), kode di sel tahap itu, dan versi pipeline. Jadi mengubah kode sebuah tahap juga membuat tahap itu dihitung ulang. Sidik disimpan di `keluaran/_meta/sNN.json` bersama daftar keluaran dan ringkasannya.

Saat **Run all**, sebuah tahap dilewati (diambil dari checkpoint) hanya bila:
1. semua berkas keluarannya masih ada,
2. sidiknya sama dengan sidik yang tersimpan, dan
3. kode tahap tidak tercantum di `paksa_ulang`.

Karena keluaran sebuah tahap menjadi masukan tahap berikutnya, perubahan di tahap awal otomatis membuat tahap-tahap sesudahnya dihitung ulang.

In [ ]:
STATUS_TAHAP = {}


def sha256_berkas(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return h.hexdigest()


def sidik_objek(obj):
    teks = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(teks.encode("utf-8")).hexdigest()


def sumber_sel_aktif():
    try:
        return get_ipython().user_ns["_ih"][-1]
    except Exception:
        return ""


def baca_csv(path):
    return pd.read_csv(path, keep_default_na=False, na_values=[""], dtype={"doc_id": str})


def simpan_csv(df, path):
    df.to_csv(path, index=False, encoding="utf-8")


class Tahap:
    def __init__(self, kode, nama, masukan, keluaran, parameter):
        self.kode = kode
        self.nama = nama
        self.masukan = [Path(p) for p in masukan]
        self.keluaran = [Path(p) for p in keluaran]
        self.parameter = parameter
        hilang = [str(p) for p in self.masukan if not p.exists()]
        if hilang:
            raise FileNotFoundError(f"[{kode}] Berkas masukan tidak ditemukan: {hilang}")
        self.sidik_masukan = {p.name: sha256_berkas(p) for p in self.masukan}
        self.sidik_kode = sidik_objek(sumber_sel_aktif())
        self.sidik = sidik_objek({
            "versi": KONFIGURASI["versi_pipeline"],
            "parameter": parameter,
            "masukan": self.sidik_masukan,
            "kode_sel": self.sidik_kode,
        })
        self.path_meta = DIR_META / f"{kode}.json"

    def bisa_dilewati(self):
        if self.kode in KONFIGURASI["paksa_ulang"]:
            return False
        if not self.path_meta.exists() or not all(p.exists() for p in self.keluaran):
            return False
        meta = json.loads(self.path_meta.read_text(encoding="utf-8"))
        if meta.get("sidik") != self.sidik:
            return False
        STATUS_TAHAP[self.kode] = "checkpoint"
        print(f"[{self.kode}] {self.nama}: diambil dari checkpoint ({meta['waktu']}).")
        for k, v in meta.get("ringkasan", {}).items():
            print(f"    {k}: {v}")
        return True

    def selesai(self, ringkasan):
        meta = {
            "kode": self.kode,
            "nama": self.nama,
            "sidik": self.sidik,
            "waktu": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "versi_pipeline": KONFIGURASI["versi_pipeline"],
            "parameter": self.parameter,
            "masukan": self.sidik_masukan,
            "keluaran": {p.name: sha256_berkas(p) for p in self.keluaran},
            "ringkasan": ringkasan,
        }
        self.path_meta.write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        STATUS_TAHAP[self.kode] = "dijalankan"
        print(f"[{self.kode}] {self.nama}: selesai dan disimpan.")
        for k, v in ringkasan.items():
            print(f"    {k}: {v}")


def tampilkan(df, n=5):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

---
# Bagian B. Korpus multi-genre

Tahap 1 sampai 5 mengolah `Combined Annotator.xlsx` menjadi berkas rilis `btb_multigenre.csv` beserta statistiknya. Hasil Tahap 3 sama persis dengan `data/btb_multigenre.csv` di repositori GitHub.

---
# Tahap 1. Muat dan validasi input anotator

**Tujuan:** memastikan berkas hasil kerja anotator terbaca utuh dan memberi setiap dokumen identitas yang tetap.

**Yang dikerjakan:**
1. Membaca `input/Combined Annotator.xlsx` (sheet pertama) apa adanya, tanpa mengubah isi teks.
2. Memeriksa bahwa kolom wajib tersedia: `Title`, `Text in Toba`, `Text in Bahasa Indonesia`, `Genre Category`, `Sub-Genre Category`.
3. Membuang baris yang benar-benar kosong (judul dan kedua teks kosong), lalu mencatat jumlahnya.
4. Memberi `doc_id` tetap dengan format `mg_001`, `mg_002`, dan seterusnya. Bila kolom `Column 1` berisi nomor urut yang unik, nomor itu dipakai. Bila tidak, dipakai urutan baris.
5. Memeriksa teks kosong, judul ganda, dan teks ganda, lalu mencatat sebaran genre dan sub-genre.
   Pemeriksaan juga mencari teks yang panjangnya tepat 32.767 karakter. Itu batas isi satu sel Excel, dan teks Agenda HKBP lebih panjang dari batas itu. Bila ada teks sepanjang itu, kemungkinan besar teksnya terpotong saat ekspor dari Google Sheet.
6. Menyimpan sidik SHA-256 berkas input agar versi data yang dipakai bisa dibuktikan.

**Masukan:** `input/Combined Annotator.xlsx`.

**Keluaran yang diharapkan:**
- `s01_dokumen_mentah.csv`: satu baris per dokumen (sekitar 104 dokumen), berisi semua kolom asli ditambah `doc_id`.
- `s01_validasi.json`: hasil pemeriksaan dan sidik berkas input.

**Dipakai untuk:** tahap 2. `doc_id` dari tahap ini dipakai sampai akhir pipeline, termasuk di `pair_id` korpus paralel.

**Cara memeriksa:** pastikan jumlah dokumen sesuai harapan, dan daftar `teks_bt_kosong` serta `teks_id_kosong` kosong. Bila tidak kosong, periksa baris tersebut di Google Sheet anotator.

**Perubahan dibanding v1:** di v1, `doc_id` bergantung pada urutan baris file v1.1 yang sudah digabung (Agenda HKBP 6 baris menjadi 1). Akibatnya `doc_0053` di korpus paralel tidak sama dengan baris 53 di `btb_multigenre.csv`. Di v2, `doc_id` mengikuti nomor dokumen anotator sehingga bisa ditelusuri langsung.

In [ ]:
P_S01_DOK = DIR_OUT / "s01_dokumen_mentah.csv"
P_S01_VAL = DIR_OUT / "s01_validasi.json"

if not BERKAS_ANOTATOR.exists():
    raise FileNotFoundError(
        f"Letakkan '{KONFIGURASI['berkas_anotator']}' di folder {DIR_INPUT} lalu jalankan ulang."
    )

t1 = Tahap(
    "s01", "Muat dan validasi input anotator",
    masukan=[BERKAS_ANOTATOR],
    keluaran=[P_S01_DOK, P_S01_VAL],
    parameter={k: KONFIGURASI[k] for k in ("sheet_anotator", "kolom_wajib")},
)

if not t1.bisa_dilewati():
    mentah = pd.read_excel(BERKAS_ANOTATOR, sheet_name=KONFIGURASI["sheet_anotator"], dtype=object)
    mentah.columns = [str(c).strip() for c in mentah.columns]

    hilang = [c for c in KONFIGURASI["kolom_wajib"] if c not in mentah.columns]
    if hilang:
        raise ValueError(f"Kolom wajib tidak ditemukan: {hilang}. Kolom yang ada: {list(mentah.columns)}")

    def kosong(nilai):
        return pd.isna(nilai) or not str(nilai).strip()

    kolom_isi = ["Title", "Text in Toba", "Text in Bahasa Indonesia"]
    baris_kosong = mentah[kolom_isi].apply(lambda r: all(kosong(v) for v in r), axis=1)
    mentah = mentah[~baris_kosong].reset_index(drop=True)

    sumber_id = "urutan_baris"
    if "Column 1" in mentah.columns:
        nomor = pd.to_numeric(mentah["Column 1"], errors="coerce")
        if nomor.notna().all() and (nomor == nomor.round()).all() and nomor.is_unique:
            mentah.insert(0, "doc_id", [f"mg_{int(n):03d}" for n in nomor])
            sumber_id = "Column 1"
    if sumber_id == "urutan_baris":
        mentah.insert(0, "doc_id", [f"mg_{i + 1:03d}" for i in range(len(mentah))])

    for kol in mentah.columns:
        mentah[kol] = mentah[kol].map(lambda v: v if pd.isna(v) else str(v))

    judul = mentah["Title"].fillna("").str.strip()
    batas_excel = 32767
    terpotong = [
        f"{r['doc_id']}:{kol}"
        for _, r in mentah.iterrows()
        for kol in ("Text in Toba", "Text in Bahasa Indonesia")
        if isinstance(r[kol], str) and len(r[kol]) == batas_excel
    ]
    validasi = {
        "berkas_input": BERKAS_ANOTATOR.name,
        "sha256_input": t1.sidik_masukan[BERKAS_ANOTATOR.name],
        "kolom_ditemukan": list(mentah.columns),
        "jumlah_dokumen": int(len(mentah)),
        "baris_kosong_dibuang": int(baris_kosong.sum()),
        "sumber_doc_id": sumber_id,
        "teks_bt_kosong": mentah.loc[mentah["Text in Toba"].map(kosong), "doc_id"].tolist(),
        "teks_id_kosong": mentah.loc[mentah["Text in Bahasa Indonesia"].map(kosong), "doc_id"].tolist(),
        "judul_ganda": sorted(judul[judul.duplicated(keep=False)].unique().tolist()),
        "teks_bt_ganda": mentah.loc[mentah["Text in Toba"].duplicated(keep=False), "doc_id"].tolist(),
        "dicurigai_terpotong_batas_excel": terpotong,
        "panjang_teks_maks": int(max(mentah["Text in Toba"].fillna("").str.len().max(),
                                     mentah["Text in Bahasa Indonesia"].fillna("").str.len().max())),
        "sebaran_genre": mentah["Genre Category"].value_counts().to_dict(),
        "sebaran_subgenre": mentah["Sub-Genre Category"].value_counts().to_dict(),
    }

    simpan_csv(mentah, P_S01_DOK)
    P_S01_VAL.write_text(json.dumps(validasi, indent=2, ensure_ascii=False), encoding="utf-8")
    t1.selesai({
        "jumlah_dokumen": validasi["jumlah_dokumen"],
        "baris_kosong_dibuang": validasi["baris_kosong_dibuang"],
        "sumber_doc_id": sumber_id,
        "teks_kosong": len(validasi["teks_bt_kosong"]) + len(validasi["teks_id_kosong"]),
        "dicurigai_terpotong": len(terpotong),
    })

dok_mentah = baca_csv(P_S01_DOK)
validasi_s01 = json.loads(P_S01_VAL.read_text(encoding="utf-8"))
print("\nSebaran genre:", validasi_s01["sebaran_genre"])
if validasi_s01["teks_bt_kosong"] or validasi_s01["teks_id_kosong"]:
    print("PERHATIAN: ada dokumen dengan teks kosong:", validasi_s01["teks_bt_kosong"], validasi_s01["teks_id_kosong"])
if validasi_s01["dicurigai_terpotong_batas_excel"]:
    print("PERHATIAN: teks berikut panjangnya tepat 32.767 karakter (batas sel Excel) dan kemungkinan terpotong:",
          validasi_s01["dicurigai_terpotong_batas_excel"])
tampilkan(dok_mentah[["doc_id", "Title", "Genre Category", "Sub-Genre Category"]])

---
# Tahap 2. Koreksi metadata tercatat

**Tujuan:** menyeragamkan struktur data dan memperbaiki kesalahan label secara terbuka, sehingga setiap perubahan bisa dipertanggungjawabkan.

**Yang dikerjakan:**
1. Mengganti nama kolom Google Sheet menjadi nama kolom baku (`title`, `text_bt`, `text_id`, `genre`, `subgenre`, `source_link`, `notes`, dan seterusnya) sesuai `kolom_anotator` di konfigurasi.
2. Menyeragamkan penulisan label, misalnya `Historical/ Traditional` menjadi `Historical/Traditional`.
3. Mengosongkan **isian contoh templat**. Enam baris pertama milik setiap anotator masih berisi metadata contoh dari templat Google Sheet (penulis "Dr. Sari Dewi", "Local Council", "Tempo Magazine"; catatan "Used for online course."; serta tanggal contoh). Nilai-nilai ini didaftar di `isian_templat` pada konfigurasi.
4. Mengosongkan `source_link` yang berisi penanda sementara seperti `[URL]`.
5. Memindahkan URL yang tertulis di kolom penulis ke kolom sumber (bila kolom sumber kosong), dan menghapus catatan yang hanya mengulang URL sumber.
6. Menerapkan daftar `koreksi_metadata`, termasuk 12 label genre atau sub-genre milik Annotator 2 yang tidak sesuai dengan daftar sumber di notebook panduan anotator (3 ulasan buku dan 9 abstrak artikel jurnal). Setiap judul dalam daftar harus cocok dengan **tepat satu** dokumen. Bila tidak, notebook berhenti dengan pesan galat agar tidak ada koreksi yang salah sasaran.
7. Memberi tanda `sumber_kosong` dan `notes_berisi_url`, lalu menampilkan semua isi `notes` yang tersisa agar peneliti bisa meninjaunya.

**Masukan:** `s01_dokumen_mentah.csv`.

**Keluaran yang diharapkan:**
- `s02_dokumen.csv`: tabel dokumen dengan kolom baku dan kolom `catatan_pipeline`.
- `s02_log_koreksi.csv`: satu baris per perubahan (doc_id, kolom, nilai lama, nilai baru, alasan).

**Dipakai untuk:** tahap 3. Log koreksi dipakai untuk bagian metodologi paper dan untuk menjawab pertanyaan reviewer tentang perubahan data.

**Cara memeriksa:** baca `s02_log_koreksi.csv`. Setiap baris harus punya alasan yang jelas. Periksa juga daftar isi `notes` yang ditampilkan di bawah sel.

**Perubahan dibanding v1:** di v1 hanya label Ikigai yang diperbaiki, itu pun hanya di v1.1 dan tidak sampai ke `btb_multigenre.csv`. Sebelas label lain yang salah, serta metadata contoh dari templat, ikut terbawa ke berkas rilis v1. Sekarang semua perbaikan itu menjadi langkah tercatat yang berlaku untuk semua keluaran.

In [ ]:
P_S02_DOK = DIR_OUT / "s02_dokumen.csv"
P_S02_LOG = DIR_OUT / "s02_log_koreksi.csv"

t2 = Tahap(
    "s02", "Koreksi metadata tercatat",
    masukan=[P_S01_DOK],
    keluaran=[P_S02_DOK, P_S02_LOG],
    parameter={k: KONFIGURASI[k] for k in ("kolom_anotator", "koreksi_metadata", "isian_templat", "normalisasi_label", "placeholder_sumber")},
)

if not t2.bisa_dilewati():
    dok = baca_csv(P_S01_DOK).rename(columns=KONFIGURASI["kolom_anotator"])
    kolom_baku = ["doc_id", "nomor_anotator", "date_collected", "title", "text_bt", "text_id", "genre",
                  "subgenre", "original_author", "source_link", "notes"]
    for kol in kolom_baku:
        if kol not in dok.columns:
            dok[kol] = np.nan
    dok = dok[kolom_baku].copy()
    dok["catatan_pipeline"] = ""
    log = []

    def catat(doc_id, kolom, lama, baru, alasan):
        log.append({"doc_id": doc_id, "kolom": kolom, "nilai_lama": lama, "nilai_baru": baru, "alasan": alasan})

    def tandai(idx, label):
        sekarang = dok.at[idx, "catatan_pipeline"]
        dok.at[idx, "catatan_pipeline"] = f"{sekarang};{label}".strip(";")

    for kol in ("title", "genre", "subgenre"):
        dok[kol] = dok[kol].fillna("").astype(str).str.strip()

    for kol, peta in KONFIGURASI["normalisasi_label"].items():
        for idx, nilai in dok[kol].items():
            if nilai in peta:
                dok.at[idx, kol] = peta[nilai]
                catat(dok.at[idx, "doc_id"], kol, nilai, peta[nilai], "Penyeragaman penulisan label")

    for kol, daftar in KONFIGURASI["isian_templat"].items():
        nilai_templat = {str(v).strip() for v in daftar}
        for idx, nilai in dok[kol].items():
            if isinstance(nilai, str) and nilai.strip() in nilai_templat:
                catat(dok.at[idx, "doc_id"], kol, nilai, "", "Isian contoh dari templat Google Sheet, bukan metadata asli")
                dok.at[idx, kol] = np.nan

    placeholder = {p.lower() for p in KONFIGURASI["placeholder_sumber"]}
    for idx, nilai in dok["source_link"].items():
        if isinstance(nilai, str) and nilai.strip().lower() in placeholder:
            catat(dok.at[idx, "doc_id"], "source_link", nilai, "", "Penanda sementara, bukan tautan sumber")
            dok.at[idx, "source_link"] = np.nan

    pola_url = re.compile(r"^\s*https?://\S+\s*$")
    for idx in dok.index:
        penulis = dok.at[idx, "original_author"]
        if isinstance(penulis, str) and pola_url.match(penulis):
            if pd.isna(dok.at[idx, "source_link"]):
                dok.at[idx, "source_link"] = penulis.strip()
                catat(dok.at[idx, "doc_id"], "source_link", "", penulis.strip(), "URL sumber tertulis di kolom penulis, dipindahkan ke kolom sumber")
            catat(dok.at[idx, "doc_id"], "original_author", penulis, "", "Kolom penulis berisi URL, bukan nama penulis")
            dok.at[idx, "original_author"] = np.nan
        catatan = dok.at[idx, "notes"]
        sumber = dok.at[idx, "source_link"]
        if isinstance(catatan, str) and isinstance(sumber, str) and catatan.strip() == sumber.strip():
            catat(dok.at[idx, "doc_id"], "notes", catatan, "", "Catatan hanya mengulang URL sumber")
            dok.at[idx, "notes"] = np.nan

    for k in KONFIGURASI["koreksi_metadata"]:
        judul_list = k["judul_mengandung"] if isinstance(k["judul_mengandung"], list) else [k["judul_mengandung"]]
        for judul in judul_list:
            cocok = dok.index[dok["title"].str.contains(judul, case=False, regex=False)]
            if len(cocok) != 1:
                raise ValueError(f"Koreksi '{judul}' cocok dengan {len(cocok)} dokumen, harus tepat 1.")
            idx = cocok[0]
            lama = dok.at[idx, k["kolom"]]
            if lama != k["nilai_baru"]:
                dok.at[idx, k["kolom"]] = k["nilai_baru"]
                catat(dok.at[idx, "doc_id"], k["kolom"], lama, k["nilai_baru"], k["alasan"])

    for idx in dok.index:
        if pd.isna(dok.at[idx, "source_link"]):
            tandai(idx, "sumber_kosong")
        if isinstance(dok.at[idx, "notes"], str) and re.search(r"https?://", dok.at[idx, "notes"]):
            tandai(idx, "notes_berisi_url")

    simpan_csv(dok, P_S02_DOK)
    log_df = pd.DataFrame(log, columns=["doc_id", "kolom", "nilai_lama", "nilai_baru", "alasan"])
    simpan_csv(log_df, P_S02_LOG)
    t2.selesai({
        "jumlah_perubahan": len(log_df),
        "dokumen_sumber_kosong": int(dok["catatan_pipeline"].str.contains("sumber_kosong").sum()),
        "dokumen_notes_berisi_url": int(dok["catatan_pipeline"].str.contains("notes_berisi_url").sum()),
    })

dok_s02 = baca_csv(P_S02_DOK)
log_s02 = baca_csv(P_S02_LOG)
tampilkan(log_s02, 20)
print("\nIsi kolom notes yang berbeda (perlu ditinjau peneliti):")
for nilai, jumlah in dok_s02["notes"].dropna().value_counts().items():
    print(f"  [{jumlah}x] {nilai}")

---
# Tahap 3. Pembersihan ringan dan berkas rilis multi-genre

**Tujuan:** merapikan aspek teknis teks tanpa mengubah kata, ejaan, kapitalisasi, atau tanda baca, lalu menghasilkan berkas rilis `btb_multigenre.csv`.

**Yang dikerjakan:**
1. Normalisasi Unicode ke bentuk NFC, sehingga huruf yang tampak sama tersimpan dengan kode yang sama.
2. Menyeragamkan akhir baris (`\r\n` menjadi `\n`), mengganti spasi tak terputus menjadi spasi biasa, dan menghapus karakter tak terlihat (zero-width).
3. Merapikan spasi berlebih dan baris kosong lebih dari dua, lalu memangkas spasi di awal dan akhir teks.
4. Mencatat dokumen mana yang berubah dan berapa karakter yang terdampak.
5. Menyusun berkas rilis dengan skema repositori: `doc_id`, `title`, `text_bt`, `text_id`, `genre`, `subgenre`, `source_link`, `notes`, `is_parallel`.

**Yang sengaja tidak dikerjakan:** huruf kecil, penghapusan tanda baca, dan perbaikan ejaan. Ketiganya mengubah data linguistik dan tidak boleh dilakukan pada berkas rilis. Versi huruf kecil tanpa tanda baca dibuat per kalimat di tahap 7 (kolom `teks_bt_norm` dan `teks_id_norm`).

**Masukan:** `s02_dokumen.csv`.

**Keluaran yang diharapkan:**
- `s03_btb_multigenre.csv`: berkas rilis korpus multi-genre, satu baris per dokumen.
- `s03_log_pembersihan.csv`: dokumen yang berubah beserta jumlah karakter sebelum dan sesudah.

**Dipakai untuk:** tahap 4 dan 5, serta disalin ke `rilis/data/btb_multigenre.csv` untuk repositori GitHub.

**Cara memeriksa:** selisih karakter per dokumen seharusnya kecil (hanya spasi dan karakter tak terlihat). Selisih besar menandakan ada masalah format di Google Sheet.

In [ ]:
P_S03_RILIS = DIR_OUT / "s03_btb_multigenre.csv"
P_S03_LOG = DIR_OUT / "s03_log_pembersihan.csv"

KARAKTER_TAK_TERLIHAT = re.compile(r"[​‌‍⁠﻿­]")


def bersihkan_teks(teks):
    if not isinstance(teks, str):
        return ""
    t = unicodedata.normalize("NFC", teks)
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = t.replace(" ", " ").replace(" ", " ")
    t = KARAKTER_TAK_TERLIHAT.sub("", t)
    t = re.sub(r"[ \t]+\n", "\n", t)
    t = re.sub(r"\n[ \t]+", "\n", t)
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()


t3 = Tahap(
    "s03", "Pembersihan ringan",
    masukan=[P_S02_DOK],
    keluaran=[P_S03_RILIS, P_S03_LOG],
    parameter={"fungsi": "bersihkan_teks v1: NFC, akhir baris, spasi, karakter tak terlihat"},
)

if not t3.bisa_dilewati():
    dok = baca_csv(P_S02_DOK)
    log = []
    for kol in ("text_bt", "text_id"):
        sebelum = dok[kol].fillna("").astype(str)
        sesudah = sebelum.map(bersihkan_teks)
        for doc_id, a, b in zip(dok["doc_id"], sebelum, sesudah):
            if a != b:
                log.append({"doc_id": doc_id, "kolom": kol, "karakter_sebelum": len(a),
                            "karakter_sesudah": len(b), "selisih": len(a) - len(b),
                            "token_spasi_sama": len(a.split()) == len(b.split())})
        dok[kol] = sesudah

    dok["is_parallel"] = np.where((dok["text_bt"].str.len() > 0) & (dok["text_id"].str.len() > 0), "yes", "no")
    rilis = dok[["doc_id", "title", "text_bt", "text_id", "genre", "subgenre", "source_link", "notes", "is_parallel"]]
    simpan_csv(rilis, P_S03_RILIS)
    log_df = pd.DataFrame(log, columns=["doc_id", "kolom", "karakter_sebelum", "karakter_sesudah", "selisih", "token_spasi_sama"])
    simpan_csv(log_df, P_S03_LOG)
    t3.selesai({
        "dokumen": len(rilis),
        "sel_teks_berubah": len(log_df),
        "total_karakter_dirapikan": int(log_df["selisih"].sum()) if len(log_df) else 0,
        "jumlah_token_spasi_berubah": int((~log_df["token_spasi_sama"]).sum()) if len(log_df) else 0,
    })

dok_rilis = baca_csv(P_S03_RILIS)
log_s03 = baca_csv(P_S03_LOG)
tampilkan(log_s03.sort_values("selisih", ascending=False) if len(log_s03) else log_s03, 10)

---
# Tahap 4. Segmentasi kalimat

**Tujuan:** memecah setiap dokumen menjadi unit kalimat (atau baris, untuk puisi, umpasa, dan teks liturgi) di kedua bahasa, sebagai bahan penyelarasan.

**Yang dikerjakan:**
1. Melindungi angka desimal (misalnya `2.5`) agar titiknya tidak dianggap akhir kalimat.
2. Memecah teks pada tanda `.`, `!`, atau `?` yang diikuti spasi dan huruf kapital atau tanda kutip pembuka.
3. Menggabungkan kembali pecahan yang terjadi setelah singkatan (misalnya `Dr.`, `Prof.`, `Op.`). Daftar singkatan ada di konfigurasi.
4. Memecah lagi pada setiap baris baru, karena puisi, umpasa, dan agenda ditulis per baris.
5. Merapikan spasi di dalam setiap kalimat.

Fungsi pemecah ini sama dengan notebook segmentasi v1, sehingga perbedaan hasil v1 dan v2 tidak berasal dari tahap ini.

**Masukan:** `s03_btb_multigenre.csv`.

**Keluaran yang diharapkan:**
- `s04_kalimat.csv`: format panjang, satu baris per kalimat (`doc_id`, `bahasa` bernilai `bbc` atau `ind`, `idx`, `teks`).
- `s04_ringkasan_segmentasi.csv`: jumlah kalimat Toba dan Indonesia per dokumen beserta selisihnya.

**Dipakai untuk:** tahap 5 (jumlah kalimat) dan tahap 6 (penyelarasan).

**Cara memeriksa:** dokumen dengan selisih jumlah kalimat besar akan lebih sulit diselaraskan dan kemungkinan besar muncul di antrean review.

In [ ]:
P_S04_KAL = DIR_OUT / "s04_kalimat.csv"
P_S04_RING = DIR_OUT / "s04_ringkasan_segmentasi.csv"

PARAM_SEG = KONFIGURASI["segmentasi"]
SINGKATAN = set(PARAM_SEG["singkatan"])
POLA_AKHIR_KALIMAT = re.compile(r'(?<=[.!?])[\s\n]+(?=[A-ZÀ-Ý"\'“‘])')
POLA_DESIMAL = re.compile(r"\d+\.\d+")


def akhir_singkatan(teks_sebelum_titik):
    kata = re.findall(r"\b\w+\b", teks_sebelum_titik.lower())
    return bool(kata) and kata[-1] in SINGKATAN


def pecah_kalimat(teks, pisah_baris=True):
    if not isinstance(teks, str) or not teks.strip():
        return []
    simpanan = {}

    def lindungi(m):
        kunci = f"__DES{len(simpanan)}__"
        simpanan[kunci] = m.group(0)
        return kunci

    terlindung = POLA_DESIMAL.sub(lindungi, teks)
    kalimat, penampung = [], ""
    for potong in POLA_AKHIR_KALIMAT.split(terlindung):
        potong = potong.strip()
        if not potong:
            continue
        if penampung:
            if akhir_singkatan(penampung.rstrip(".!?").rstrip()):
                penampung = penampung + " " + potong
                continue
            kalimat.append(penampung)
        penampung = potong
    if penampung:
        kalimat.append(penampung)

    hasil = []
    for k in kalimat:
        bagian = re.split(r"\n+", k) if pisah_baris else re.split(r"\n{2,}", k)
        hasil.extend(b.strip() for b in bagian if b.strip())
    hasil = [re.sub(r"__DES\d+__", lambda m: simpanan.get(m.group(0), m.group(0)), k) for k in hasil]
    hasil = [re.sub(r"\s+", " ", k).strip() for k in hasil]
    return [k for k in hasil if k]


t4 = Tahap(
    "s04", "Segmentasi kalimat",
    masukan=[P_S03_RILIS],
    keluaran=[P_S04_KAL, P_S04_RING],
    parameter=PARAM_SEG,
)

if not t4.bisa_dilewati():
    dok = baca_csv(P_S03_RILIS)
    baris, ringkasan = [], []
    for _, r in dok.iterrows():
        jumlah = {}
        for bahasa, kol in (("bbc", "text_bt"), ("ind", "text_id")):
            daftar = pecah_kalimat(r[kol], PARAM_SEG["pisah_setiap_baris_baru"])
            jumlah[bahasa] = len(daftar)
            baris.extend({"doc_id": r["doc_id"], "bahasa": bahasa, "idx": i, "teks": k} for i, k in enumerate(daftar))
        ringkasan.append({"doc_id": r["doc_id"], "title": r["title"], "genre": r["genre"], "subgenre": r["subgenre"],
                          "kalimat_bt": jumlah["bbc"], "kalimat_id": jumlah["ind"],
                          "selisih": abs(jumlah["bbc"] - jumlah["ind"])})
    kal = pd.DataFrame(baris)
    ring = pd.DataFrame(ringkasan)
    simpan_csv(kal, P_S04_KAL)
    simpan_csv(ring, P_S04_RING)
    t4.selesai({
        "kalimat_bt": int(ring["kalimat_bt"].sum()),
        "kalimat_id": int(ring["kalimat_id"].sum()),
        "dokumen_jumlah_kalimat_sama": int((ring["selisih"] == 0).sum()),
    })

kalimat = baca_csv(P_S04_KAL)
ring_seg = baca_csv(P_S04_RING)
tampilkan(ring_seg.sort_values("selisih", ascending=False), 10)

---
# Tahap 5. Statistik korpus

**Tujuan:** menghasilkan angka resmi korpus dengan metode hitung yang **dinyatakan secara eksplisit**, sehingga angka di README, laporan teknis, dan paper konsisten dan bisa diperiksa ulang.

**Yang dikerjakan:**
1. Menghitung token dengan tiga metode untuk setiap dokumen dan bahasa:
   - `token_spasi`: pemisahan berdasarkan spasi pada teks rilis. Tanda baca yang menempel pada kata tidak dihitung terpisah.
   - `kata`: hanya urutan huruf atau angka (`\w+`). Tanda baca tidak dihitung. Ini angka yang paling tepat disebut "jumlah kata".
   - `token_nltk_v1`: metode EDA versi lama, yaitu teks dijadikan huruf kecil, dibersihkan, lalu dipecah dengan NLTK `word_tokenize`. **Tanda baca ikut dihitung sebagai token.** Metode ini disertakan hanya agar angka lama (112.340 token) bisa direproduksi dan dibandingkan.
2. Menghitung jumlah kalimat dari hasil tahap 4.
3. Merangkum per genre dan sub-genre, termasuk porsi teks religius dan porsi Agenda HKBP.
4. Memeriksa target 100.000 token untuk setiap metode secara terpisah.
5. Menyusun potongan tabel Markdown siap tempel untuk README.

**Masukan:** `s03_btb_multigenre.csv` dan `s04_ringkasan_segmentasi.csv`.

**Keluaran yang diharapkan:**
- `s05_statistik.json`: total per metode, status target, porsi genre.
- `s05_statistik_per_genre.csv` dan `s05_statistik_per_dokumen.csv`.
- `s05_tabel_readme.md`: tabel siap pakai untuk README dan laporan teknis.

**Dipakai untuk:** README repositori, laporan teknis, bagian statistik korpus di paper.

**Cara memeriksa:** bandingkan `token_nltk_v1` dengan angka lama 67.680 (sisi Toba multi-genre). Bila sama, metode lama berhasil direproduksi. Saat melaporkan, sebutkan metode yang dipakai.

In [ ]:
P_S05_JSON = DIR_OUT / "s05_statistik.json"
P_S05_GENRE = DIR_OUT / "s05_statistik_per_genre.csv"
P_S05_DOK = DIR_OUT / "s05_statistik_per_dokumen.csv"
P_S05_MD = DIR_OUT / "s05_tabel_readme.md"

import nltk
for paket in ("punkt", "punkt_tab"):
    try:
        nltk.download(paket, quiet=True)
    except Exception as e:
        print("Gagal mengunduh", paket, e)
from nltk.tokenize import sent_tokenize, word_tokenize


def bersihkan_gaya_eda_v1(teks):
    t = teks.lower()
    t = re.sub(r"[^a-z0-9.,!?;:\s]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()


def token_nltk_v1(teks):
    t = bersihkan_gaya_eda_v1(teks)
    return sum(len(word_tokenize(k)) for p in t.split("\n") for k in sent_tokenize(p)) if t else 0


t5 = Tahap(
    "s05", "Statistik korpus",
    masukan=[P_S03_RILIS, P_S04_RING],
    keluaran=[P_S05_JSON, P_S05_GENRE, P_S05_DOK, P_S05_MD],
    parameter={"target_token": KONFIGURASI["target_token"], "metode": ["token_spasi", "kata", "token_nltk_v1"],
               "subgenre_liturgi": KONFIGURASI["subgenre_liturgi"]},
)

if not t5.bisa_dilewati():
    dok = baca_csv(P_S03_RILIS)
    seg = baca_csv(P_S04_RING)[["doc_id", "kalimat_bt", "kalimat_id"]]
    rows = []
    for _, r in dok.iterrows():
        baris = {"doc_id": r["doc_id"], "title": r["title"], "genre": r["genre"], "subgenre": r["subgenre"]}
        for suf, kol in (("bt", "text_bt"), ("id", "text_id")):
            teks = r[kol] if isinstance(r[kol], str) else ""
            baris[f"token_spasi_{suf}"] = len(teks.split())
            baris[f"kata_{suf}"] = len(re.findall(r"\w+", teks))
            baris[f"token_nltk_v1_{suf}"] = token_nltk_v1(teks)
        rows.append(baris)
    per_dok = pd.DataFrame(rows).merge(seg, on="doc_id", how="left")
    metrik = [c for c in per_dok.columns if c.startswith(("token_", "kata_", "kalimat_"))]

    per_genre = per_dok.groupby("genre")[metrik].sum()
    per_genre.insert(0, "dokumen", per_dok.groupby("genre").size())
    per_genre["porsi_kata_bt_persen"] = (100 * per_genre["kata_bt"] / per_genre["kata_bt"].sum()).round(2)
    per_genre = per_genre.reset_index()

    total = {c: int(per_dok[c].sum()) for c in metrik}
    target = KONFIGURASI["target_token"]
    liturgi = per_dok["subgenre"].isin(KONFIGURASI["subgenre_liturgi"])
    religius = per_dok["genre"] == "Religious"
    statistik = {
        "jumlah_dokumen": int(len(per_dok)),
        "total": total,
        "target_token": target,
        "target_tercapai_sisi_bt": {
            "token_spasi": total["token_spasi_bt"] >= target,
            "kata": total["kata_bt"] >= target,
            "token_nltk_v1": total["token_nltk_v1_bt"] >= target,
        },
        "porsi_religius_kata_bt_persen": round(100 * per_dok.loc[religius, "kata_bt"].sum() / total["kata_bt"], 2),
        "porsi_liturgi_kata_bt_persen": round(100 * per_dok.loc[liturgi, "kata_bt"].sum() / total["kata_bt"], 2),
        "catatan_metode": {
            "token_spasi": "split() pada teks rilis; tanda baca menempel pada kata",
            "kata": "regex \\w+; tanda baca tidak dihitung",
            "token_nltk_v1": "huruf kecil, pembersihan gaya EDA v1, NLTK word_tokenize; tanda baca dihitung sebagai token",
        },
    }

    def fmt(n):
        return f"{n:,}".replace(",", ".")

    tabel = [
        "| Ukuran (sub-korpus multi-genre) | Batak Toba | Bahasa Indonesia |",
        "|---|---|---|",
        f"| Dokumen | {fmt(len(per_dok))} | {fmt(len(per_dok))} |",
        f"| Kalimat atau baris (segmentasi tahap 4) | {fmt(total['kalimat_bt'])} | {fmt(total['kalimat_id'])} |",
        f"| Kata (tanpa tanda baca) | {fmt(total['kata_bt'])} | {fmt(total['kata_id'])} |",
        f"| Token berbasis spasi | {fmt(total['token_spasi_bt'])} | {fmt(total['token_spasi_id'])} |",
        f"| Token NLTK, termasuk tanda baca (metode v1) | {fmt(total['token_nltk_v1_bt'])} | {fmt(total['token_nltk_v1_id'])} |",
    ]
    simpan_csv(per_dok, P_S05_DOK)
    simpan_csv(per_genre, P_S05_GENRE)
    P_S05_JSON.write_text(json.dumps(statistik, indent=2, ensure_ascii=False), encoding="utf-8")
    P_S05_MD.write_text("\n".join(tabel) + "\n", encoding="utf-8")
    t5.selesai({
        "kata_bt": total["kata_bt"],
        "token_spasi_bt": total["token_spasi_bt"],
        "token_nltk_v1_bt": total["token_nltk_v1_bt"],
        "porsi_religius_persen": statistik["porsi_religius_kata_bt_persen"],
    })

statistik_s05 = json.loads(P_S05_JSON.read_text(encoding="utf-8"))
print(P_S05_MD.read_text(encoding="utf-8"))
print("Target tercapai (sisi Batak Toba):", statistik_s05["target_tercapai_sisi_bt"])
print("Porsi religius:", statistik_s05["porsi_religius_kata_bt_persen"], "% | porsi liturgi (Agenda HKBP):",
      statistik_s05["porsi_liturgi_kata_bt_persen"], "%")
tampilkan(baca_csv(P_S05_GENRE), 10)

---
# Bagian C. Korpus paralel tingkat kalimat

Bagian ini mengolah korpus paralel v1 dan hasil anotasi dua penilai menjadi korpus paralel final.

| Tahap | Isi | Keluaran utama |
|---|---|---|
| 6 | Muat dan validasi korpus paralel v1 serta lembar anotasi | `s06_korpus_v1.csv`, `s06_lembar_anotasi.csv` |
| 7 | Rekonsiliasi label dan kesepakatan antaranotator (IAA) | `s07_laporan_iaa.json`, `s07_matriks_konfusi.csv`, `s07_sensitivitas.csv` |
| 8 | Keputusan emas dari adjudikasi | `s08_keputusan_emas.csv` |
| 9 | Penerapan keputusan emas ke korpus | `s09_korpus_paralel_final.csv`, `s09_verifikasi.json` |

---
## Tahap 6. Muat dan validasi korpus paralel v1 serta lembar anotasi

**Tujuan:** memastikan korpus v1 dan ketiga lembar anotasi terbaca utuh, merujuk pada pasangan yang sama, dan hanya memakai kategori yang sah.

**Latar belakang:** korpus paralel v1 (1.505 pasangan) dihasilkan sebelum notebook ini dibuat, melalui segmentasi, penyelarasan LaBSE, dan audit penyelarasan semi-manual. Sebagian langkahnya, termasuk pengurangan dari 1.562 menjadi 1.494 pasangan sebelum audit, tidak tercatat di notebook mana pun. Karena itu korpus v1 dipakai sebagai **input beku**: isinya tidak diubah, hanya divalidasi.

**Yang dikerjakan:**
1. Membaca `btb_parallel_sentence_v1.csv` dan memeriksa bahwa `pair_id` unik dan teks tidak kosong.
2. Membaca lembar `Perlu_periksa` dari ketiga berkas anotasi (A asli, A rekonsiliasi, B).
3. Memeriksa bahwa ketiganya memuat 145 baris yang sama dengan urutan yang sama, dan semuanya ada di korpus v1.
4. Memeriksa bahwa teks asli di lembar anotasi sama dengan teks di korpus v1.
5. Memeriksa bahwa setiap keputusan berasal dari enam kategori yang sah.
6. Menggabungkan ketiga lembar menjadi satu tabel kerja.

**Masukan:** `btb_parallel_sentence_v1.csv`, `anotator_A_asli.xlsx`, `anotator_A_rekonsiliasi.xlsx`, `anotator_B.xlsx`.

**Keluaran yang diharapkan:**
- `s06_korpus_v1.csv`: salinan korpus v1 (1.505 baris).
- `s06_lembar_anotasi.csv`: 145 baris berisi teks asli, prioritas, serta keputusan dan teks koreksi dari A asli, A rekonsiliasi, dan B.
- `s06_validasi.json`: hasil pemeriksaan.

**Dipakai untuk:** tahap 7 (IAA) dan tahap 8 (keputusan emas).

**Kategori keputusan anotasi:**

| Kategori | Arti |
|---|---|
| `accept_unit` | Pasangan sudah benar dan tidak perlu diubah |
| `revise_alignment` | Batas unit salah. Unit perlu dipecah atau digeser, dengan `|||` sebagai penanda pecah |
| `revise_translation` | Batas unit benar, tetapi terjemahannya perlu diperbaiki |
| `need_source` | Tidak bisa diputuskan tanpa teks asli atau konteks lebih luas |
| `fragment_policy` | Pasangan sepadan tetapi bukan kalimat lengkap (judul, butir daftar) |
| `exclude` | Pasangan dikeluarkan dari korpus |

In [ ]:
P_S06_V1 = DIR_OUT / "s06_korpus_v1.csv"
P_S06_LEMBAR = DIR_OUT / "s06_lembar_anotasi.csv"
P_S06_VAL = DIR_OUT / "s06_validasi.json"
PARAM_IAA = KONFIGURASI["iaa"]


def bersih(x):
    return "" if pd.isna(x) else re.sub(r"\s+", " ", str(x)).strip()


t6 = Tahap(
    "s06", "Muat dan validasi korpus v1 serta lembar anotasi",
    masukan=[BERKAS["korpus_paralel_v1"], BERKAS["anotator_A_asli"], BERKAS["anotator_A_rekonsiliasi"], BERKAS["anotator_B"]],
    keluaran=[P_S06_V1, P_S06_LEMBAR, P_S06_VAL],
    parameter={"sheet": KONFIGURASI["sheet_review"], "kategori": PARAM_IAA["kategori"]},
)

if not t6.bisa_dilewati():
    v1 = pd.read_csv(BERKAS["korpus_paralel_v1"], dtype=str, keep_default_na=False)
    if not v1["pair_id"].is_unique:
        raise ValueError("pair_id di korpus v1 tidak unik.")
    kosong_v1 = int(((v1["toba_final"].str.strip() == "") | (v1["indo_final"].str.strip() == "")).sum())

    lembar = {}
    for kode, kunci in (("A_asli", "anotator_A_asli"), ("A", "anotator_A_rekonsiliasi"), ("B", "anotator_B")):
        d = pd.read_excel(BERKAS[kunci], sheet_name=KONFIGURASI["sheet_review"], dtype=str)
        d["annotator_decision"] = d["annotator_decision"].map(bersih)
        asing = sorted(set(d["annotator_decision"]) - set(PARAM_IAA["kategori"]))
        if asing:
            raise ValueError(f"{kunci} memuat kategori tidak dikenal: {asing}")
        lembar[kode] = d

    urutan = lembar["A"]["pair_id"].tolist()
    for kode, d in lembar.items():
        if d["pair_id"].tolist() != urutan:
            raise ValueError(f"Urutan pair_id lembar {kode} berbeda dengan lembar A rekonsiliasi.")
    tidak_ada = sorted(set(urutan) - set(v1["pair_id"]))
    if tidak_ada:
        raise ValueError(f"pair_id berikut tidak ada di korpus v1: {tidak_ada[:5]}")

    idx_v1 = v1.set_index("pair_id")
    beda_teks = [p for p, b, i in zip(urutan, lembar["A"]["toba_final"], lembar["A"]["indo_final"])
                 if (bersih(b), bersih(i)) != (bersih(idx_v1.at[p, "toba_final"]), bersih(idx_v1.at[p, "indo_final"]))]

    tabel = pd.DataFrame({
        "pair_id": urutan,
        "doc_id_v1": lembar["A"]["doc_id"],
        "prioritas": lembar["A"]["review_priority"],
        "teks_bt_asli": [bersih(idx_v1.at[p, "toba_final"]) for p in urutan],
        "teks_id_asli": [bersih(idx_v1.at[p, "indo_final"]) for p in urutan],
    })
    for kode, d in lembar.items():
        tabel[f"keputusan_{kode}"] = d["annotator_decision"].values
        tabel[f"koreksi_bt_{kode}"] = d["corrected_toba"].map(bersih).values
        tabel[f"koreksi_id_{kode}"] = d["corrected_indo"].map(bersih).values

    validasi = {
        "pasangan_v1": int(len(v1)),
        "pasangan_v1_teks_kosong": kosong_v1,
        "baris_anotasi": int(len(tabel)),
        "teks_asli_lembar_berbeda_dari_v1": beda_teks,
        "sebaran_prioritas": tabel["prioritas"].value_counts().to_dict(),
        "sebaran_keputusan": {k: tabel[f"keputusan_{k}"].value_counts().to_dict() for k in lembar},
    }
    simpan_csv(v1, P_S06_V1)
    simpan_csv(tabel, P_S06_LEMBAR)
    P_S06_VAL.write_text(json.dumps(validasi, indent=2, ensure_ascii=False), encoding="utf-8")
    t6.selesai({
        "pasangan_v1": validasi["pasangan_v1"],
        "baris_anotasi": validasi["baris_anotasi"],
        "teks_lembar_beda_dari_v1": len(beda_teks),
    })

lembar_anotasi = baca_csv(P_S06_LEMBAR)
validasi_s06 = json.loads(P_S06_VAL.read_text(encoding="utf-8"))
print("\nSebaran keputusan per lembar:")
tampilkan(pd.DataFrame(validasi_s06["sebaran_keputusan"]).fillna(0).astype(int), 10)

---
## Tahap 7. Rekonsiliasi label dan kesepakatan antaranotator (IAA)

**Tujuan:** mengukur seberapa konsisten dua penilai menilai 145 pasangan yang sama, dan memeriksa bahwa rekonsiliasi label Anotator A dilakukan sesuai aturan.

**Latar belakang rekonsiliasi:** Anotator A semula memberi `accept_unit` pada 123 baris, tetapi menyunting teks pada 92 di antaranya. A menafsirkan `accept_unit` sebagai "pasangan ini sah untuk disimpan", bukan "tidak perlu diubah". A kemudian diminta merekonsiliasi labelnya dengan aturan: kalau teks diubah, keputusannya bukan `accept_unit`, melainkan `revise_alignment` (bila unit dipecah) atau `revise_translation`. Revisi ini didasarkan pada suntingan A sendiri, bukan pada hasil Anotator B.

**Yang dikerjakan:**
1. **Memeriksa rekonsiliasi.** Aturan di atas diterapkan secara otomatis ke lembar A asli, lalu hasilnya dibandingkan dengan lembar A rekonsiliasi. Bila keduanya sama persis, rekonsiliasi terbukti mekanis dan konsisten.
2. **Menghitung kesepakatan** (persentase dan Cohen's kappa enam kategori) untuk empat skenario:

| Skenario | A | B | Keterangan |
|---|---|---|---|
| `asli` | asli | asli | Sebelum rekonsiliasi |
| `rekonsiliasi_A` | rekonsiliasi | asli | Skenario yang menjadi dasar adjudikasi |
| `simetris` | aturan diterapkan | aturan diterapkan | Aturan yang sama juga diterapkan ke B, yang punya baris `accept_unit` dengan teks tersunting |
| `simetris_abaikan_tanda_baca` | aturan diterapkan | aturan diterapkan | Seperti `simetris`, tetapi suntingan yang hanya mengubah spasi, tanda baca, atau huruf besar tidak dihitung |

3. Untuk skenario utama (`kappa_utama` di konfigurasi): interval kepercayaan 95% dengan bootstrap, kappa biner (`accept_unit` dibandingkan dengan selain itu), kesepakatan per prioritas, dan matriks konfusi.

**Interpretasi kappa** (Landis dan Koch, 1977): di bawah 0 buruk, 0,00 sampai 0,20 sangat lemah (*slight*), 0,21 sampai 0,40 lemah (*fair*), 0,41 sampai 0,60 sedang, 0,61 sampai 0,80 kuat, 0,81 sampai 1,00 hampir sempurna.

**Masukan:** `s06_lembar_anotasi.csv`.

**Keluaran yang diharapkan:**
- `s07_laporan_iaa.json`: semua angka kesepakatan.
- `s07_sensitivitas.csv`: tabel keempat skenario.
- `s07_matriks_konfusi.csv`: keputusan A dibandingkan dengan B pada skenario utama.

**Dipakai untuk:** bagian validasi data di paper.

**Cara memeriksa:** pastikan `rekonsiliasi_sesuai_aturan` bernilai `True`, dan kappa skenario `rekonsiliasi_A` sama dengan angka di laporan IAA (0,170).

In [ ]:
P_S07_JSON = DIR_OUT / "s07_laporan_iaa.json"
P_S07_SENS = DIR_OUT / "s07_sensitivitas.csv"
P_S07_KONF = DIR_OUT / "s07_matriks_konfusi.csv"


def kappa_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return float("nan")
    po = float(np.mean(a == b))
    pe = sum(float(np.mean(a == k)) * float(np.mean(b == k)) for k in set(a) | set(b))
    return 1.0 if pe == 1 else (po - pe) / (1 - pe)


def interval_bootstrap(a, b, n, seed):
    a, b = np.asarray(a), np.asarray(b)
    rng = np.random.default_rng(seed)
    nilai = [kappa_cohen(a[i], b[i]) for i in (rng.integers(0, len(a), len(a)) for _ in range(n))]
    nilai = np.array([v for v in nilai if not np.isnan(v)])
    return [round(float(np.percentile(nilai, 2.5)), 4), round(float(np.percentile(nilai, 97.5)), 4)]


def tafsir_kappa(k):
    for batas, label in ((0, "buruk"), (0.20, "sangat lemah (slight)"), (0.40, "lemah (fair)"), (0.60, "sedang (moderate)"), (0.80, "kuat (substantial)")):
        if k <= batas:
            return label
    return "hampir sempurna"


def normal_ringan(teks):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s|]", "", str(teks).lower())).strip()


def rekonsiliasi(df, kode, abaikan_tanda_baca=False):
    f = normal_ringan if abaikan_tanda_baca else (lambda x: x)
    tersunting = (df[f"koreksi_bt_{kode}"].map(f) != df["teks_bt_asli"].map(f)) | (df[f"koreksi_id_{kode}"].map(f) != df["teks_id_asli"].map(f))
    pecah = df[f"koreksi_bt_{kode}"].str.contains(re.escape(PARAM_IAA["penanda_pecah"])) | df[f"koreksi_id_{kode}"].str.contains(re.escape(PARAM_IAA["penanda_pecah"]))
    hasil = df[f"keputusan_{kode}"].copy()
    ubah = (hasil == "accept_unit") & tersunting
    hasil[ubah] = np.where(pecah[ubah], "revise_alignment", "revise_translation")
    return hasil, int(ubah.sum())


def ringkas(a, b):
    k = kappa_cohen(a, b)
    return {"n": int(len(a)), "kesepakatan_persen": round(100 * float(np.mean(np.asarray(a) == np.asarray(b))), 1),
            "kappa": round(k, 3), "tafsir": tafsir_kappa(k)}


t7 = Tahap(
    "s07", "Rekonsiliasi label dan IAA",
    masukan=[P_S06_LEMBAR],
    keluaran=[P_S07_JSON, P_S07_SENS, P_S07_KONF],
    parameter={**PARAM_IAA, "seed": KONFIGURASI["seed"]},
)

if not t7.bisa_dilewati():
    df = baca_csv(P_S06_LEMBAR).fillna("")
    for kol in df.columns:
        if kol.startswith("koreksi_") or kol.startswith("teks_"):
            df[kol] = df[kol].map(bersih)

    A_aturan, n_ubah_A = rekonsiliasi(df, "A_asli")
    B_aturan, n_ubah_B = rekonsiliasi(df, "B")
    A_aturan_tb, _ = rekonsiliasi(df, "A_asli", abaikan_tanda_baca=True)
    B_aturan_tb, n_ubah_B_tb = rekonsiliasi(df, "B", abaikan_tanda_baca=True)
    rekonsiliasi_sesuai = bool((A_aturan == df["keputusan_A"]).all())
    beda_rekonsiliasi = df.loc[A_aturan != df["keputusan_A"], "pair_id"].tolist()

    skenario = {
        "asli": (df["keputusan_A_asli"], df["keputusan_B"]),
        "rekonsiliasi_A": (df["keputusan_A"], df["keputusan_B"]),
        "simetris": (A_aturan, B_aturan),
        "simetris_abaikan_tanda_baca": (A_aturan_tb, B_aturan_tb),
    }
    sens = pd.DataFrame([{"skenario": k, **ringkas(a, b)} for k, (a, b) in skenario.items()])

    utama = PARAM_IAA["kappa_utama"]
    a, b = (np.asarray(x) for x in skenario[utama])
    biner = lambda x: np.where(x == "accept_unit", "accept_unit", "bukan_accept")
    laporan = {
        "rekonsiliasi_sesuai_aturan": rekonsiliasi_sesuai,
        "rekonsiliasi_baris_berubah_A": int((df["keputusan_A_asli"] != df["keputusan_A"]).sum()),
        "rekonsiliasi_baris_tidak_sesuai_aturan": beda_rekonsiliasi,
        "B_accept_unit_dengan_teks_tersunting": n_ubah_B,
        "B_accept_unit_tersunting_bukan_hanya_tanda_baca": n_ubah_B_tb,
        "skenario_utama": utama,
        "utama": {
            **ringkas(a, b),
            "kappa_ik95": interval_bootstrap(a, b, PARAM_IAA["bootstrap"], KONFIGURASI["seed"]),
            "kappa_biner_accept": round(kappa_cohen(biner(a), biner(b)), 3),
            "baris_sepakat": int((a == b).sum()),
            "baris_tidak_sepakat": int((a != b).sum()),
        },
        "per_prioritas": {p: ringkas(a[(df["prioritas"] == p).values], b[(df["prioritas"] == p).values])
                          for p in sorted(df["prioritas"].unique())},
        "sebaran_keputusan_A": pd.Series(a).value_counts().to_dict(),
        "sebaran_keputusan_B": pd.Series(b).value_counts().to_dict(),
        "sensitivitas": sens.to_dict(orient="records"),
    }
    konfusi = pd.crosstab(pd.Series(a, name="A"), pd.Series(b, name="B")).reindex(
        index=PARAM_IAA["kategori"], columns=PARAM_IAA["kategori"], fill_value=0)

    P_S07_JSON.write_text(json.dumps(laporan, indent=2, ensure_ascii=False), encoding="utf-8")
    simpan_csv(sens, P_S07_SENS)
    konfusi.to_csv(P_S07_KONF, encoding="utf-8")
    t7.selesai({
        "rekonsiliasi_sesuai_aturan": rekonsiliasi_sesuai,
        "kappa_utama": laporan["utama"]["kappa"],
        "kappa_ik95": laporan["utama"]["kappa_ik95"],
        "baris_tidak_sepakat": laporan["utama"]["baris_tidak_sepakat"],
    })

laporan_iaa = json.loads(P_S07_JSON.read_text(encoding="utf-8"))
print("\nAnalisis sensitivitas:")
tampilkan(baca_csv(P_S07_SENS), 10)
print("\nMatriks konfusi (baris: A, kolom: B):")
tampilkan(pd.read_csv(P_S07_KONF, index_col=0), 10)

---
## Tahap 8. Keputusan emas dari adjudikasi

**Tujuan:** menetapkan satu keputusan akhir (keputusan emas) beserta teks finalnya untuk setiap dari 145 pasangan.

**Yang dikerjakan:** setiap baris mendapat keputusan emas dari tepat satu sumber berikut, diperiksa dengan urutan ini:

| Sumber | Kapan dipakai | Keputusan dan teks diambil dari |
|---|---|---|
| `koreksi_pasca_adjudikasi` | Baris tercantum di `koreksi_pasca_adjudikasi` di konfigurasi | Konfigurasi, lengkap dengan alasan dan sumbernya |
| `adjudikasi_keputusan` | A dan B berbeda keputusan (78 baris) | Kolom `FINAL_keputusan`, `FINAL_corr_toba`, `FINAL_corr_indo` |
| `adjudikasi_teks` | A dan B sepakat `revise_*` tetapi teks koreksinya berbeda (49 baris) | Kolom `gold_decision`, `FINAL_corr_toba`, `FINAL_corr_indo` |
| `sepakat` | A dan B sepakat dan teks koreksinya sama, atau sepakat `accept_unit` | Lembar A rekonsiliasi |

Notebook juga memeriksa bahwa lembar adjudikasi keputusan memuat tepat baris-baris yang tidak disepakati, dan lembar adjudikasi teks memuat tepat baris yang disepakati dengan teks berbeda. Kalau tidak, notebook berhenti.

**Masukan:** `s06_lembar_anotasi.csv`, `s07_laporan_iaa.json`, `adjudikasi_keputusan_78.xlsx`, `adjudikasi_teks_49.xlsx`.

**Keluaran yang diharapkan:** `s08_keputusan_emas.csv`, 145 baris berisi `pair_id`, keputusan emas, teks final Toba dan Indonesia, sumber keputusan, serta catatan adjudikasi.

**Dipakai untuk:** tahap 9.

In [ ]:
P_S08_EMAS = DIR_OUT / "s08_keputusan_emas.csv"

t8 = Tahap(
    "s08", "Keputusan emas dari adjudikasi",
    masukan=[P_S06_LEMBAR, BERKAS["adjudikasi_keputusan"], BERKAS["adjudikasi_teks"]],
    keluaran=[P_S08_EMAS],
    parameter={"koreksi_pasca_adjudikasi": KONFIGURASI["koreksi_pasca_adjudikasi"], "kategori": PARAM_IAA["kategori"]},
)

if not t8.bisa_dilewati():
    df = baca_csv(P_S06_LEMBAR).fillna("").set_index("pair_id")
    adj_k = pd.read_excel(BERKAS["adjudikasi_keputusan"], sheet_name=0, dtype=str).set_index("pair_id")
    adj_t = pd.read_excel(BERKAS["adjudikasi_teks"], sheet_name=0, dtype=str).set_index("pair_id")

    tidak_sepakat = set(df.index[df["keputusan_A"] != df["keputusan_B"]])
    sepakat_teks_beda = set(df.index[(df["keputusan_A"] == df["keputusan_B"]) & df["keputusan_A"].str.startswith("revise")
                                     & ((df["koreksi_bt_A"] != df["koreksi_bt_B"]) | (df["koreksi_id_A"] != df["koreksi_id_B"]))])
    if set(adj_k.index) != tidak_sepakat:
        raise ValueError("Lembar adjudikasi keputusan tidak memuat tepat baris yang tidak disepakati.")
    if set(adj_t.index) != sepakat_teks_beda:
        raise ValueError("Lembar adjudikasi teks tidak memuat tepat baris sepakat dengan teks koreksi berbeda.")

    pasca = {k["pair_id"]: k for k in KONFIGURASI["koreksi_pasca_adjudikasi"]}
    baris = []
    for pid, r in df.iterrows():
        if pid in adj_k.index:
            sumber, kep = "adjudikasi_keputusan", adj_k.at[pid, "FINAL_keputusan"]
            bt, idn, cat = adj_k.at[pid, "FINAL_corr_toba"], adj_k.at[pid, "FINAL_corr_indo"], adj_k.at[pid, "catatan_adjudikasi"]
        elif pid in adj_t.index:
            sumber, kep = "adjudikasi_teks", adj_t.at[pid, "gold_decision"]
            bt, idn, cat = adj_t.at[pid, "FINAL_corr_toba"], adj_t.at[pid, "FINAL_corr_indo"], adj_t.at[pid, "catatan"]
        else:
            sumber, kep, bt, idn, cat = "sepakat", r["keputusan_A"], r["koreksi_bt_A"], r["koreksi_id_A"], ""
        keputusan_adjudikasi = kep
        if pid in pasca:
            k = pasca[pid]
            sumber, kep = "koreksi_pasca_adjudikasi", k["keputusan"]
            bt = k["teks_bt"] if k["teks_bt"] is not None else r["teks_bt_asli"]
            idn = k["teks_id"] if k["teks_id"] is not None else r["teks_id_asli"]
            cat = f"{k['alasan']} Sumber: {k['sumber']}."
        if bersih(kep) not in PARAM_IAA["kategori"]:
            raise ValueError(f"{pid}: keputusan emas tidak dikenal: {kep}")
        baris.append({"pair_id": pid, "prioritas": r["prioritas"], "keputusan_A": r["keputusan_A"], "keputusan_B": r["keputusan_B"],
                      "keputusan_adjudikasi": bersih(keputusan_adjudikasi), "keputusan_emas": bersih(kep),
                      "teks_bt_final": bersih(bt), "teks_id_final": bersih(idn), "sumber_keputusan": sumber, "catatan": bersih(cat)})
    emas = pd.DataFrame(baris)
    simpan_csv(emas, P_S08_EMAS)
    t8.selesai({
        "baris": len(emas),
        "sumber": emas["sumber_keputusan"].value_counts().to_dict(),
        "keputusan_emas": emas["keputusan_emas"].value_counts().to_dict(),
    })

keputusan_emas = baca_csv(P_S08_EMAS)
for k in KONFIGURASI["koreksi_pasca_adjudikasi"]:
    if "BELUM" in k["sumber"].upper():
        print(f"PERHATIAN: sumber koreksi pasca-adjudikasi untuk {k['pair_id']} belum dikonfirmasi. Lengkapi kolom 'sumber' di Konfigurasi.")
print(pd.crosstab(keputusan_emas["sumber_keputusan"], keputusan_emas["keputusan_emas"]))

---
## Tahap 9. Penerapan keputusan emas ke korpus

**Tujuan:** menghasilkan korpus paralel final dengan menerapkan keputusan emas ke korpus v1, lalu menyelaraskan `doc_id` dan label genrenya dengan korpus multi-genre.

**Aturan penerapan:**

| Keputusan emas | Yang dilakukan pada pasangan | `review_status` |
|---|---|---|
| (tidak ditandai) | Dipertahankan apa adanya | `not_flagged` |
| `accept_unit` | Teks asli dipertahankan, meskipun kolom koreksi berisi teks | `accepted` |
| `revise_translation`, `fragment_policy`, `need_source` | Teks diganti teks final. Bila memuat `|||`, unit dipecah | `revised` atau `revised_split` |
| `revise_alignment` | Unit dipecah pada `|||`. Bila penanda itu tidak ada atau jumlah segmen kedua sisi berbeda, sisi yang kurang dipecah per kalimat | `revised_split` (atau `revised` bila tetap satu unit) |
| `exclude` | Pasangan dibuang | (tidak masuk korpus) |

Pasangan hasil pemecahan diberi `pair_id` baru dengan akhiran `__r01`, `__r02`, dan seterusnya. Semua teks dirapikan spasinya.

**Penyelarasan dengan korpus multi-genre:** korpus v1 masih memakai `doc_id` lama (`doc_0000` sampai `doc_0097`, urutan berkas v1.1 yang menggabungkan enam bagian Agenda HKBP) dan 11 label genre yang sudah dikoreksi di tahap 2. Tahap ini menambahkan `doc_id` baru (`mg_001` sampai `mg_104`) dan mengambil genre serta sub-genre dari hasil tahap 3. Pemetaan `doc_id` diperiksa dengan mencocokkan teks: setiap teks Toba yang tidak ditandai harus ditemukan di dokumen multi-genre hasil pemetaan.

**Verifikasi:** bila `korpus_utama_FINAL_v3` tersedia, hasil tahap ini (sebelum `doc_id` dan genre diganti) dibandingkan sel demi sel dengan berkas itu. Kolom `alignment_type` dibandingkan terpisah, karena Google Sheets mengubah nilai seperti `1-1` menjadi tanggal.

**Masukan:** `s06_korpus_v1.csv`, `s08_keputusan_emas.csv`, `s03_btb_multigenre.csv`, dan opsional `korpus_utama_FINAL_v3`.

**Keluaran yang diharapkan:**
- `s09_korpus_paralel_final.csv`: korpus paralel final (sekitar 1.542 pasangan).
- `s09_verifikasi.json`: hasil pemeriksaan pemetaan dan perbandingan dengan FINAL_v3.

**Dipakai untuk:** tahap 10 dan rilis di `data/btb_parallel_sentence.csv`.

In [ ]:
P_S09_FINAL = DIR_OUT / "s09_korpus_paralel_final.csv"
P_S09_VER = DIR_OUT / "s09_verifikasi.json"
POLA_KALIMAT_ADJ = re.compile(r'(?<=[.!?])\s+(?=[A-Z"“])')
P_FINAL_V3 = BERKAS["pembanding_final_v3"]
if not P_FINAL_V3.exists() and P_FINAL_V3.with_suffix(".xlsx").exists():
    P_FINAL_V3 = P_FINAL_V3.with_suffix(".xlsx")


def segmen(teks):
    return [x for x in (bersih(p) for p in str(teks).split(PARAM_IAA["penanda_pecah"])) if x]


def segmen_alignment(bt, idn, pid):
    sb, si = segmen(bt), segmen(idn)
    if len(sb) != len(si) or len(sb) == 1:
        if len(sb) < len(si) or (len(sb) == 1 and len(si) == 1):
            sb = [x for s in sb for x in POLA_KALIMAT_ADJ.split(s)]
        if len(si) < len(sb) or len(si) == 1:
            si = [x for s in si for x in POLA_KALIMAT_ADJ.split(s)]
    if len(sb) != len(si):
        raise ValueError(f"{pid}: jumlah segmen Toba ({len(sb)}) dan Indonesia ({len(si)}) tidak sama.")
    return sb, si


def peta_doc_id(doc_v1):
    p = KONFIGURASI["peta_doc_id_v1"]
    i = int(doc_v1.split("_")[1])
    return f"mg_{i + 1 if i < p['batas_agenda'] else i + 1 + p['geser_setelah_agenda']:03d}"


masukan_s09 = [P_S06_V1, P_S08_EMAS, P_S03_RILIS] + ([P_FINAL_V3] if P_FINAL_V3.exists() else [])
t9 = Tahap(
    "s09", "Penerapan keputusan emas ke korpus",
    masukan=masukan_s09,
    keluaran=[P_S09_FINAL, P_S09_VER],
    parameter={"peta": KONFIGURASI["peta_doc_id_v1"], "penanda": PARAM_IAA["penanda_pecah"]},
)

if not t9.bisa_dilewati():
    v1 = baca_csv(P_S06_V1).fillna("")
    emas = baca_csv(P_S08_EMAS).fillna("").set_index("pair_id")
    kolom_v3 = ["pair_id", "doc_id", "genre", "sub_genre", "method", "alignment_type", "toba_final", "indo_final", "source"]
    baris = []
    for r in v1.itertuples(index=False):
        dasar = {k: bersih(getattr(r, k)) for k in kolom_v3}
        if r.pair_id not in emas.index:
            baris.append({**dasar, "review_status": "not_flagged", "gold_decision": ""})
            continue
        e = emas.loc[r.pair_id]
        kep = e["keputusan_emas"]
        if kep == "exclude":
            continue
        if kep == "accept_unit":
            baris.append({**dasar, "review_status": "accepted", "gold_decision": kep})
            continue
        if kep == "revise_alignment":
            sb, si = segmen_alignment(e["teks_bt_final"], e["teks_id_final"], r.pair_id)
        else:
            sb, si = segmen(e["teks_bt_final"]) or [""], segmen(e["teks_id_final"]) or [""]
            if len(sb) != len(si):
                raise ValueError(f"{r.pair_id}: jumlah segmen Toba dan Indonesia tidak sama.")
        if len(sb) == 1:
            baris.append({**dasar, "toba_final": sb[0], "indo_final": si[0], "review_status": "revised", "gold_decision": kep})
        else:
            for k, (x, y) in enumerate(zip(sb, si), start=1):
                baris.append({**dasar, "pair_id": f"{r.pair_id}__r{k:02d}", "toba_final": x, "indo_final": y,
                              "review_status": "revised_split", "gold_decision": kep})
    final = pd.DataFrame(baris)

    verifikasi = {"pasangan_final": int(len(final))}
    if P_FINAL_V3.exists():
        v3 = (pd.read_csv(P_FINAL_V3, dtype=str, keep_default_na=False) if P_FINAL_V3.suffix == ".csv"
              else pd.read_excel(P_FINAL_V3, dtype=str).fillna(""))
        v3 = v3.map(bersih)
        sama_baris = len(v3) == len(final) and v3["pair_id"].tolist() == final["pair_id"].tolist()
        kolom_banding = [c for c in final.columns if c in v3.columns and c != "alignment_type"]
        beda = {}
        if sama_baris:
            d = final[kolom_banding].astype(str).values != v3[kolom_banding].astype(str).values
            beda = {c: int(n) for c, n in zip(kolom_banding, d.sum(axis=0)) if n}
            beda_at = int((final["alignment_type"].values != v3["alignment_type"].values).sum())
            beda_at_bukan_tanggal = int(((final["alignment_type"].values != v3["alignment_type"].values)
                                         & ~v3["alignment_type"].str.match(r"^\d{4}-\d{2}-\d{2}").values).sum())
        verifikasi["pembanding_final_v3"] = {
            "berkas": P_FINAL_V3.name,
            "baris_dan_urutan_pair_id_sama": sama_baris,
            "sel_berbeda_per_kolom": beda,
            "alignment_type_berbeda": beda_at if sama_baris else None,
            "alignment_type_berbeda_bukan_karena_tanggal": beda_at_bukan_tanggal if sama_baris else None,
            "identik": bool(sama_baris and not beda and beda_at_bukan_tanggal == 0),
        }
    else:
        verifikasi["pembanding_final_v3"] = "berkas pembanding tidak tersedia"

    mg = baca_csv(P_S03_RILIS).fillna("").set_index("doc_id")
    final.insert(1, "doc_id_v1", final["doc_id"])
    agenda = KONFIGURASI["peta_doc_id_v1"]["doc_agenda_v1"]
    if (final["doc_id_v1"] == agenda).any():
        raise ValueError("Korpus v1 memuat dokumen Agenda HKBP gabungan yang tidak bisa dipetakan.")
    final["doc_id"] = final["doc_id_v1"].map(peta_doc_id)
    padat = lambda s: re.sub(r"\s+", "", s)
    cek = final[final["review_status"] == "not_flagged"]
    ketemu = [padat(t) in padat(mg.at[d, "text_bt"]) for t, d in zip(cek["toba_final"], cek["doc_id"])]
    per_dok = pd.Series(ketemu, index=cek["doc_id"].values).groupby(level=0).mean()
    verifikasi["pemetaan_doc_id"] = {
        "dokumen": int(final["doc_id"].nunique()),
        "proporsi_teks_ditemukan": round(float(np.mean(ketemu)), 4),
        "dokumen_proporsi_di_bawah_80_persen": per_dok[per_dok < 0.8].round(3).to_dict(),
    }
    if per_dok.min() < 0.8:
        raise ValueError(f"Pemetaan doc_id meragukan: {per_dok[per_dok < 0.8].to_dict()}")
    genre_lama = final["genre"] + " / " + final["sub_genre"]
    final["genre"] = final["doc_id"].map(mg["genre"])
    final["sub_genre"] = final["doc_id"].map(mg["subgenre"])
    berubah = final.loc[genre_lama != final["genre"] + " / " + final["sub_genre"]]
    verifikasi["label_genre_diperbarui"] = {
        "pasangan": int(len(berubah)),
        "dokumen": sorted(berubah["doc_id"].unique().tolist()),
    }
    final["validasi_manusia"] = np.where(final["review_status"] == "not_flagged", "tidak", "ya")
    final = final[["pair_id", "doc_id", "doc_id_v1", "genre", "sub_genre", "method", "alignment_type", "toba_final",
                   "indo_final", "source", "review_status", "gold_decision", "validasi_manusia"]]

    simpan_csv(final, P_S09_FINAL)
    P_S09_VER.write_text(json.dumps(verifikasi, indent=2, ensure_ascii=False), encoding="utf-8")
    t9.selesai({
        "pasangan_final": len(final),
        "identik_dengan_final_v3": verifikasi["pembanding_final_v3"]["identik"] if isinstance(verifikasi["pembanding_final_v3"], dict) else None,
        "review_status": final["review_status"].value_counts().to_dict(),
        "pasangan_label_genre_diperbarui": verifikasi["label_genre_diperbarui"]["pasangan"],
    })

korpus_paralel = baca_csv(P_S09_FINAL)
verifikasi_s09 = json.loads(P_S09_VER.read_text(encoding="utf-8"))
print(json.dumps(verifikasi_s09, indent=2, ensure_ascii=False))
tampilkan(korpus_paralel[korpus_paralel["review_status"] == "revised_split"], 6)

---
# Bagian D. Statistik akhir dan paket rilis

## Tahap 10. Statistik korpus paralel dan paket rilis

**Tujuan:** menghitung statistik korpus paralel final dan mengumpulkan semua berkas final ke satu folder yang siap disalin ke repositori GitHub.

**Yang dikerjakan:**
1. Menghitung jumlah pasangan, dokumen, kata Toba dan Indonesia, serta jumlah pasangan yang divalidasi manusia, per genre dan per status.
2. Menyalin berkas final ke `rilis/` dengan struktur folder repositori.
3. Menulis `manifest.json` (versi pustaka, konfigurasi, sidik SHA-256 input dan keluaran) dan `CHANGELOG.md`.

**Keluaran yang diharapkan:**

```
rilis/
├── data/
│   ├── btb_multigenre.csv
│   └── btb_parallel_sentence.csv
├── iaa/
│   ├── laporan_iaa.json
│   ├── sensitivitas.csv
│   ├── matriks_konfusi.csv
│   └── keputusan_emas.csv
├── statistik/
│   ├── statistik_korpus_multigenre.json
│   ├── statistik_per_genre_multigenre.csv
│   ├── statistik_korpus_paralel.json
│   ├── statistik_per_genre_paralel.csv
│   └── tabel_readme.md
├── log/
│   ├── log_koreksi_metadata.csv
│   ├── log_pembersihan.csv
│   └── verifikasi_korpus_paralel.json
├── manifest.json
└── CHANGELOG.md
```

**Dipakai untuk:** pembaruan repositori GitHub, README, laporan teknis, dan lampiran reproduktibilitas paper.

In [ ]:
P_S10_JSON = DIR_OUT / "s10_statistik_paralel.json"
P_S10_GENRE = DIR_OUT / "s10_statistik_per_genre_paralel.csv"


def jumlah_kata(teks):
    return len(re.findall(r"\w+", str(teks)))


t10 = Tahap(
    "s10", "Statistik korpus paralel dan paket rilis",
    masukan=[P_S09_FINAL, P_S05_JSON, P_S07_JSON],
    keluaran=[P_S10_JSON, P_S10_GENRE],
    parameter={},
)

if not t10.bisa_dilewati():
    kp = baca_csv(P_S09_FINAL).fillna("")
    kp["kata_bt"] = kp["toba_final"].map(jumlah_kata)
    kp["kata_id"] = kp["indo_final"].map(jumlah_kata)
    per_genre = kp.groupby(["genre", "sub_genre"]).agg(
        dokumen=("doc_id", "nunique"), pasangan=("pair_id", "size"),
        divalidasi_manusia=("validasi_manusia", lambda s: int((s == "ya").sum())),
        kata_bt=("kata_bt", "sum"), kata_id=("kata_id", "sum")).reset_index()
    stat = {
        "pasangan": int(len(kp)),
        "dokumen": int(kp["doc_id"].nunique()),
        "kata_bt": int(kp["kata_bt"].sum()),
        "kata_id": int(kp["kata_id"].sum()),
        "review_status": kp["review_status"].value_counts().to_dict(),
        "pasangan_divalidasi_manusia": int((kp["validasi_manusia"] == "ya").sum()),
        "persen_divalidasi_manusia": round(100 * float((kp["validasi_manusia"] == "ya").mean()), 1),
        "gold_decision": kp.loc[kp["gold_decision"] != "", "gold_decision"].value_counts().to_dict(),
    }
    P_S10_JSON.write_text(json.dumps(stat, indent=2, ensure_ascii=False), encoding="utf-8")
    simpan_csv(per_genre, P_S10_GENRE)
    t10.selesai({k: stat[k] for k in ("pasangan", "dokumen", "kata_bt", "pasangan_divalidasi_manusia")})

stat_paralel = json.loads(P_S10_JSON.read_text(encoding="utf-8"))
statistik_mg = json.loads(P_S05_JSON.read_text(encoding="utf-8"))
iaa = json.loads(P_S07_JSON.read_text(encoding="utf-8"))


def fmt(n):
    return f"{n:,}".replace(",", ".")


tabel_md = P_S05_MD.read_text(encoding="utf-8") + "\n" + "\n".join([
    "| Ukuran (korpus paralel tingkat kalimat) | Nilai |",
    "|---|---|",
    f"| Pasangan | {fmt(stat_paralel['pasangan'])} |",
    f"| Dokumen | {fmt(stat_paralel['dokumen'])} |",
    f"| Kata Batak Toba / Indonesia | {fmt(stat_paralel['kata_bt'])} / {fmt(stat_paralel['kata_id'])} |",
    f"| Pasangan divalidasi manusia | {fmt(stat_paralel['pasangan_divalidasi_manusia'])} ({str(stat_paralel['persen_divalidasi_manusia']).replace('.', ',')}%) |",
    f"| Cohen's kappa ({iaa['skenario_utama']}, n = {iaa['utama']['n']}) | {str(iaa['utama']['kappa']).replace('.', ',')} ({iaa['utama']['tafsir']}) |",
]) + "\n"

salinan = {
    P_S03_RILIS: "data/btb_multigenre.csv",
    P_S09_FINAL: "data/btb_parallel_sentence.csv",
    P_S07_JSON: "iaa/laporan_iaa.json",
    P_S07_SENS: "iaa/sensitivitas.csv",
    P_S07_KONF: "iaa/matriks_konfusi.csv",
    P_S08_EMAS: "iaa/keputusan_emas.csv",
    P_S05_JSON: "statistik/statistik_korpus_multigenre.json",
    P_S05_GENRE: "statistik/statistik_per_genre_multigenre.csv",
    P_S10_JSON: "statistik/statistik_korpus_paralel.json",
    P_S10_GENRE: "statistik/statistik_per_genre_paralel.csv",
    P_S02_LOG: "log/log_koreksi_metadata.csv",
    P_S03_LOG: "log/log_pembersihan.csv",
    P_S09_VER: "log/verifikasi_korpus_paralel.json",
}
for asal, tujuan in salinan.items():
    (DIR_RILIS / tujuan).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(asal, DIR_RILIS / tujuan)
(DIR_RILIS / "statistik" / "tabel_readme.md").write_text(tabel_md, encoding="utf-8")


def versi_pustaka(nama):
    try:
        from importlib.metadata import version
        return version(nama)
    except Exception:
        return "tidak terpasang"


manifest = {
    "nama": "Korpus Batak Toba dan Indonesia",
    "versi_pipeline": KONFIGURASI["versi_pipeline"],
    "dibuat": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "lingkungan": {"python": platform.python_version(), "colab": DI_COLAB,
                   "pustaka": {p: versi_pustaka(p) for p in ("pandas", "numpy", "nltk", "openpyxl")}},
    "input": {str(p.relative_to(DIR_INPUT)): sha256_berkas(p) for p in BERKAS.values() if p.exists()},
    "konfigurasi": KONFIGURASI,
    "status_tahap": STATUS_TAHAP,
    "berkas_rilis": {str(p.relative_to(DIR_RILIS)): {"sha256": sha256_berkas(p), "baris": int(len(baca_csv(p))) if p.suffix == ".csv" else None}
                     for p in sorted(DIR_RILIS.rglob("*")) if p.is_file() and p.name not in ("manifest.json", "CHANGELOG.md")},
}
(DIR_RILIS / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

tot = statistik_mg["total"]
ver = json.loads(P_S09_VER.read_text(encoding="utf-8"))
changelog = [
    "# CHANGELOG",
    "",
    f"Dibuat otomatis oleh notebook alur lengkap v{KONFIGURASI['versi_pipeline']} pada {manifest['dibuat']}.",
    "",
    "## Korpus multi-genre",
    "",
    f"- Dokumen: {fmt(statistik_mg['jumlah_dokumen'])}",
    f"- Kata Batak Toba (tanpa tanda baca): {fmt(tot['kata_bt'])}",
    f"- Token Batak Toba NLTK termasuk tanda baca (metode lama): {fmt(tot['token_nltk_v1_bt'])}",
    "- Dua belas label genre atau sub-genre dikoreksi sesuai panduan anotator, metadata contoh templat dikosongkan.",
    "",
    "## Korpus paralel tingkat kalimat",
    "",
    f"- Pasangan: {fmt(stat_paralel['pasangan'])} dari {fmt(stat_paralel['dokumen'])} dokumen",
    f"- Pasangan divalidasi manusia: {fmt(stat_paralel['pasangan_divalidasi_manusia'])}",
    f"- Status: {stat_paralel['review_status']}",
    f"- Identik dengan korpus_utama_FINAL_v3: {ver['pembanding_final_v3']['identik'] if isinstance(ver['pembanding_final_v3'], dict) else 'tidak diperiksa'}",
    f"- doc_id disamakan dengan korpus multi-genre (mg_NNN); label genre diperbarui pada {fmt(ver['label_genre_diperbarui']['pasangan'])} pasangan.",
    "",
    "## Kesepakatan antaranotator",
    "",
    f"- Skenario utama {iaa['skenario_utama']}: n = {iaa['utama']['n']}, kesepakatan {iaa['utama']['kesepakatan_persen']}%, "
    f"kappa {iaa['utama']['kappa']} ({iaa['utama']['tafsir']}), IK95% {iaa['utama']['kappa_ik95']}",
] + [f"- Skenario {s['skenario']}: kesepakatan {s['kesepakatan_persen']}%, kappa {s['kappa']}" for s in iaa["sensitivitas"]]
(DIR_RILIS / "CHANGELOG.md").write_text("\n".join(changelog) + "\n", encoding="utf-8")
STATUS_TAHAP["s10"] = STATUS_TAHAP.get("s10", "dijalankan")

print(tabel_md)
print("Rilis tersimpan di", DIR_RILIS)
for p in sorted(DIR_RILIS.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(DIR_RILIS))

---
# Bagian E. Ringkasan dan keterbatasan

Sel berikut menampilkan status setiap tahap dan angka kunci. Salin tabel ini ke catatan penelitian setiap kali notebook dijalankan ulang.

In [ ]:
baris = []
for kode in ["s01", "s02", "s03", "s04", "s05", "s06", "s07", "s08", "s09", "s10"]:
    meta_path = DIR_META / f"{kode}.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    baris.append({"tahap": kode, "nama": meta.get("nama", ""), "status": STATUS_TAHAP.get(kode, "belum dijalankan"),
                  "waktu": meta.get("waktu", ""), "ringkasan": json.dumps(meta.get("ringkasan", {}), ensure_ascii=False)})
tampilkan(pd.DataFrame(baris), 20)

## Keterbatasan yang perlu disebut di paper

1. **Validasi manusia terbatas.** Hanya 145 pasangan (berprioritas tinggi atau sedang) yang dinilai dua penilai dan diadjudikasi. Pasangan lainnya (`review_status = not_flagged`) adalah hasil penyelarasan otomatis yang lolos audit, belum diperiksa manusia. Korpus paralel sebaiknya tidak disebut korpus emas secara keseluruhan. Kolom `validasi_manusia` menandai pasangan mana yang sudah divalidasi.
2. **IAA hanya mengukur kasus sulit.** Karena sampel penilaian hanya berisi pasangan yang ditandai bermasalah, angka kesepakatan tidak mewakili kualitas korpus secara keseluruhan.
3. **Kesepakatan rendah.** Kappa pada kisaran 0,16 sampai 0,18 di semua skenario menunjukkan kedua penilai belum memahami batas antarkategori secara sama, terutama antara `revise_translation`, `revise_alignment`, dan `fragment_policy`. Ini keterbatasan pedoman kategori, bukan bukti tentang kualitas korpus.
4. **Rekonsiliasi label hanya dilakukan pada satu penilai.** Analisis sensitivitas di tahap 7 menunjukkan dampaknya bila aturan yang sama diterapkan ke kedua penilai. Adjudikasi tetap didasarkan pada skenario `rekonsiliasi_A`.
5. **Adjudikasi oleh satu orang.** Tidak ada ukuran reliabilitas untuk tahap adjudikasi. Satu pasangan (`doc_0000_s013`) diterjemahkan ulang oleh peneliti setelah adjudikasi dan tercatat di `koreksi_pasca_adjudikasi`.
6. **Korpus v1 adalah input beku.** Sebagian langkah pembentukannya (penyelarasan dan audit awal) tidak bisa dijalankan ulang dari notebook ini.
7. **Agenda HKBP tidak masuk korpus paralel.** Dokumen liturgi ini (59% kata Toba korpus multi-genre) hanya ada di korpus multi-genre.
8. **Domain religius dominan** di korpus multi-genre, dan sub-korpus Alkitab belum disertakan.